<div align="center">

# Hafta 1: Python ve kardiyoloji veri temelleri

**Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı**<br>
Türk Kardiyoloji Derneği · Prof. Dr. Utku Köse

[Hafta sayfası](README.md) · [Ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html) · [Etkileşimli lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/lab.html) · [PDF ders notu](Hafta01_Ders_Notlari.pdf) · [Colab'da aç](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/NB01_python_ve_ekg_temelleri.ipynb)

</div>

**Bu Jupyter Notebook nasıl kullanılır?** Jupyter Notebook, ders sayfasındaki uygulama duraklarını izler. Her durak üç bölümden oluşur ve her bölüm tek başına da çalışır: Python temelleri, hazır prompt (istem) metinleriyle uygulama ve hazır uygulama. **Python temelleri** bölümü, durağın gerektirdiği Python bilgisini haftanın konusundan alınan küçük örneklerle öğretir; bu bölümdeki alıştırmalar sonucu kendileri denetler. **Prompt ile uygulama** bölümünde bir yapay zekâ (YZ) asistanı için hazır promptlar bulunur: Kodu asistan yazar, siz çalıştırırsınız ve bir kontrol hücresi sonucun doğru olup olmadığını gösterir. **Hazır uygulama** bölümü ise durağın baştan sona çalışan Python programıdır ve kodu açıklayan notlar içerir.

Üç bölümü bu sırayla izleyin. Python biliyorsanız doğrudan promptlarla başlayabilir, yalnızca sonucu görmek istiyorsanız hazır uygulamayı açabilirsiniz. Sondaki haftalık proje de kendi başına çalışır.

**Bir bölüme başlamak.** Bölümün ilk hücresini, yani hazırlık hücresini çalıştırın. Bu hücre, 0. bölüm o oturumda çalışmadıysa onu ve bölümün ihtiyaç duyduğu önceki kodları da çalıştırır; böylece yukarıdaki hücreleri aramanız gerekmez. En üstteki 0. bölüm, Jupyter Notebook'u baştan izleyenler için ders dosyalarını ve paketleri hazırlar.

**Veri.** Bir veri setini okuyan her hücre üç kaynak sunar: `"kütüphane"` veriyi bir Python kütüphanesiyle veri setinin kendi sitesinden okur, `"web"` dosyayı eğitimin GitHub deposundaki kopyadan okur, `"bilgisayar"` ise sizin indirdiğiniz dosyayı okur. Google Colab'da kaynağı hücrenin sağındaki kutudan seçin; Jupyter'de `VERI_KAYNAGI = ...` satırını değiştirin. `"bilgisayar"` seçildiğinde Jupyter dosyayı bu Jupyter Notebook'un klasöründe arar, Colab ise dosyayı yüklemeniz için bir pencere açar.

**Yerinizi bulmak.** İçindekiler paneli her durağı ve her bölümü listeler. Bu panel Colab'da soldaki liste simgesiyle, Jupyter'de View → Table of Contents menüsüyle açılır. Her bölüm, ders sayfasına dönen bir bağlantıyla biter. Colab'da yaptığınız değişiklikler ancak Drive'a kaydedilen bir kopyada kalıcı olur: File → Save a copy in Drive komutunu kullanın.

**Nerede çalışır?** Jupyter Notebook, kendi bilgisayarınızdaki Jupyter'de (Anaconda) ve Google Colab'da çalışır. Buradaki bütün veri setleri ya sentetiktir ya da açık bir araştırma veri tabanından gelir. Gerçek hasta verisi yalnızca kurumunuzun bir bilgisayarında ve kurumunuzun kurallarına göre işlenir; bulut tabanlı bir Jupyter Notebook'a ya da bir YZ sohbetine asla konmaz.

## 0. Kurulum

**Jupyter Notebook'un ilk kullanımı.** Bir Jupyter Notebook hücrelerden oluşur. Bu hücre metin içerir; aşağıdaki gri hücrelerde Python kodu vardır. Bir kod hücresinin içine tıklayın ve Shift+Enter tuşlarına basın: Hücre çalışır, sonucu hücrenin altında görünür ve solundaki köşeli parantez içindeki sayı, örneğin `[1]`, çalıştırma sırasını gösterir. Jupyter Notebook'un arkasındaki Python süreci olan kernel, hücrelerin tanımladığı her adı hatırlar; bu yüzden hücreler yukarıdan aşağıya çalıştırılır. **Kernel → Restart Kernel and Run All Cells** komutu Jupyter Notebook'un tamamını yeniden çalıştırır, Ctrl+S (Mac'te Cmd+S) ise Jupyter Notebook'u `.ipynb` dosyası olarak kaydeder.

İlk hücre ders klasörünü bulur. Jupyter Notebook tek başına açılmışsa (örneğin Google Colab'da), hücre `vclinic` ders paketini ve veri dosyalarını GitHub'dan bir kez indirir. İkinci hücre paketleri yükler ve mesajların dilini seçer. Ayrıca haftanın iki EKG paketi olan `wfdb` ve `neurokit2` paketlerini denetler; eksiklerse onları pip ile bir kez kurar.

In [ ]:
#@title Ders dosyaları: Bu hücreyi çalıştırın (açmanıza gerek yok)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
    "data/uci-heart-disease/processed.cleveland.data",
    "data/physionet/mitdb/100.hea",
    "data/physionet/mitdb/100.dat",
    "data/physionet/mitdb/100.atr",
    "data/physionet/ptbdb/patient001/s0010_re.hea",
    "data/physionet/ptbdb/patient001/s0010_re.dat",
    "data/physionet/ptbdb/patient001/s0010_re.xyz",
    "data/examples/example_caliper_measurements.csv",
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # İndirilmiş bir kopyada, örneğin Colab'da, paketin dosyaları yenilenir.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Ders dosyaları şu klasöre indirildi:", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Kurulum: paketler, dil ve çizim biçimi
import importlib
import importlib.util

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.set_language("tr")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026

print("vclinic sürümü", vclinic.__version__)
print("Ders klasörü:", ROOT)
import subprocess
import sys

for name in ["wfdb", "neurokit2", "ipywidgets"]:
    try:
        importlib.import_module(name)
        print(f"{name}: kurulu")
    except ImportError:
        if name in ("wfdb", "neurokit2"):
            # haftanın iki EKG paketi pip ile bir kez kurulur
            result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", name], capture_output=True, text=True)
            importlib.invalidate_caches()
            ok = result.returncode == 0 and importlib.util.find_spec(name) is not None
            print(f"{name}: " + ("şimdi kuruldu" if ok else "kurulamadı (internet yok mu?); Jupyter Notebook yerleşik bir yedek yöntem kullanır"))
        else:
            print(f"{name}: kurulu değil (isteğe bağlı)")

In [ ]:
#@title Alıştırmaların ve promptların beklenen cevapları (bu hücreyi çalıştırın; açmayı alıştırmaları denedikten sonraya bırakın)
teach.register_answers({
    "Alıştırma 1.1": (0.833, 0.001),
    "Alıştırma 1.2": (439.1, 0.2),
    "Alıştırma 1.3": (2, 0),
    "Alıştırma 1.4": (64.5, 0.05),
    "Alıştırma 2.1": (100, 0),
    "Alıştırma 2.2": (0.104, 0.002),
    "Alıştırma 2.3": (216, 6),
    "Alıştırma 3.1": (39.05, 0.1),
    "Alıştırma 3.2": (808.0, 0.05),
    "Prompt 1.1": (56.6, 0.05),
    "Prompt 1.2": (32.0, 0.05),
    "Prompt 1.3": (47, 0),
    "Prompt 2.1": (1250, 1),
    "Prompt 2.2": (84.9, 0.5),
    "Prompt 2.3": (403.9, 0.05),
    "Prompt 3.1": (33, 0),
    "Prompt 3.2": (36.0, 0.1),
    "Prompt 3.3": (81.8, 0.5),
    "Prompt 3.4": (466.5, 0.05),
})
print("Beklenen cevaplar hazır.")

## Durak 1: Mini proje: Kaggle Heart Disease veri seti

*Ders sayfasında: [1. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#stop-1)*

Bu durak oturumun mini projesidir: UCI Heart Disease koleksiyonunun Cleveland verisi olan Kaggle Heart Disease veri seti yüklenir, istatistiksel olarak özetlenir ve görselleştirilir. Bir kliniğin hasta kaydı bir tablo olarak düşünülebilir: Her hasta bir satırda, her değişken bir sütunda yer alır. Bu durakta veri setindeki 303 hasta ile sanal kliniğin ilk 500 hastası bir ortalama, bir oran ve bir sayıyla özetlenmektedir. Bu sayıların her birinin ardında klinik bir soru vardır.

**Durağın sonunda:** Koroner arter hastalığı olan Cleveland hastalarının ortalama yaşını (56,6 yıl), 303 hastanın tamamında kadınların oranını (yüzde 32,0) ve sanal klinikteki HFrEF hastalarının sayısını (47) elde edeceksiniz. Bunlara, ejeksiyon fraksiyonunu tanılara göre gösteren bir kutu grafiği eşlik eder.

Durak üç bölümden oluşur: **Python temelleri**, **Prompt ile uygulama** ve **Hazır uygulama**. Bölümleri bu sırayla izleyin. Python biliyorsanız promptlarla, yalnızca sonucu görmek istiyorsanız hazır uygulamayla başlayabilirsiniz. Her bölümün ilk hücresi, bölümün ihtiyaç duyduğu her şeyi hazırlar.

### Durak 1 · Python temelleri

*Ders sayfasında: [Değişkenler, listeler, döngüler: Tıbbi örneklerle](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#variables-lists-and-loops-with-medical-examples)*

Python bu eğitimin dilidir. Bir kod hücresi Python komutlarını içerir ve Shift+Enter bu komutları çalıştırır. `#` işaretinden sonra gelen metin yorumdur: Python onu atlar, okur ise kodun açıklamasını orada bulur. Bu bölümdeki her örnek hücre `# Python alıştırması: dili öğrenmek için küçük bir örnek` satırıyla başlar. Örneklerde haftanın vakasındaki sayılar kullanılmaktadır: Vaka, kalp hızı dakikada 85 atım ve QT aralığı 391 ms olan 67 yaşında bir kadın hastadır.

Hücreleri sırayla çalıştırın. Bir örneğin altındaki *Python notu* yeni yapıyı açıklar. Alıştırmalar hemen *doğru* ya da *henüz değil* yazar; örnek çözümler katlanmış hücrelerdedir.

In [ ]:
#@title Python temellerinin hazırlığı: Bu hücreyi bir kez çalıştırın (0. bölüm bu oturumda çalışmadıysa onu çalıştırır)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
print("1. durağın Python temelleri hazır.")

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
heart_rate = 85          # kalp hızı, atım/dk
qt_ms = 391              # milisaniye cinsinden QT aralığı
rr_s = 60 / heart_rate   # saniye cinsinden RR aralığı
print(rr_s)
print(round(rr_s, 3))

**Python notu.** `heart_rate = 85` satırı bir *değişken* oluşturur; değişken, bir değeri tutan addır. `=` işareti atama yapar, eşitliği sınamaz. `/` bölme işlemidir ve `60 / heart_rate` RR aralığını saniye cinsinden verir. `print()` bir değeri ekrana yazar, `round(değer, 3)` ise değeri üç ondalık basamağa yuvarlar. Python kayan noktalı sayılarla hesap yaptığı için sonuç, yuvarlanmadan önce `0.7058823529411765` biçiminde bütün basamaklarıyla görünür. Python'da ondalık ayırıcı her zaman noktadır.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
patient_id = "VC-0317"
print(f"Hasta {patient_id}: {heart_rate} atım/dk, RR {rr_s:.3f} s")

**Python notu.** Tırnak içindeki metne *karakter dizisi* (string) denir. Önünde `f` harfi bulunan karakter dizisine *f-string* denir: Süslü parantezlerin içindeki değerler metnin içine yerleştirilir ve `{rr_s:.3f}` sayıyı üç ondalık basamakla yazar.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
troponin = [3.6, 4.1, 3.9]          # seri ölçümler (ng/L)
print(troponin[0], troponin[-1], len(troponin))
troponin.append(4.0)                # dördüncü ölçüm geliyor
print(troponin, "max:", max(troponin))

**Python notu.** Köşeli parantezler bir *liste* oluşturur; liste, değerlerin sıralı bir dizisidir. Python konumları 0'dan saymaya başlar; bu yüzden `troponin[0]` ilk değeri, `troponin[-1]` ise son değeri verir. `len()` eleman sayısını verir, `.append()` listenin sonuna bir eleman ekler, `max()` ise en büyük değeri döndürür.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
patient = {"id": "VC-0317", "age": 67, "sex": "F", "hr_bpm": 85, "qt_ms": 391}
print(patient["age"])
patient["lvef"] = 63                # yeni alan: sol ventrikül ejeksiyon fraksiyonu (%)
print(patient)

**Python notu.** `anahtar: değer` çiftleri içeren süslü parantezler bir *sözlük* (dictionary) oluşturur. Sözlük bir hasta dosyası gibi çalışır: Her alanın bir etiketi vardır. `patient["age"]` bir alanı okur, `patient["lvef"] = 63` ise yeni bir alan ekler.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
qtc_ms = 465
limit = 460 if patient["sex"] == "F" else 450
if qtc_ms >= limit:
    print(f"QTc değeri {qtc_ms} ms; bu değer {limit} ms'lik sınırda ya da sınırın üzerindedir")
else:
    print(f"QTc değeri {qtc_ms} ms; bu değer {limit} ms'lik sınırın altındadır")

**Python notu.** `if`, altındaki girintili satırları yalnızca koşul doğru olduğunda çalıştırır; `else` öteki durumu karşılar. Girinti (dört boşluk) hangi satırın hangi dala ait olduğunu gösterir. `==` eşitliği sınar, `>=` ise "büyük ya da eşit" anlamına gelir. `limit = 460 if ... else 450` satırı iki değerden birini tek satırda seçer. Bu iki sınır, Amerikan Kalp Derneği (American Heart Association, AHA) ve ortaklarının 2009 önerilerinde kadınlar ve erkekler için uzamış QTc eşikleridir.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
clinic = [
    {"id": "VC-0317", "hr": 85, "qt": 391},
    {"id": "VC-0079", "hr": 58, "qt": 404},
    {"id": "VC-0254", "hr": 98, "qt": 343},
]
for p in clinic:
    rr = 60 / p["hr"]
    print(p["id"], round(rr, 3))

**Python notu.** `clinic`, her hasta için bir sözlük içeren bir listedir. `for p in clinic:` satırı hastaları tek tek alır ve girintili satırlar her hasta için bir kez çalışır; tıpkı hasta hasta ilerleyen bir servis vizitesi gibi.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
def qtc_bazett(qt_ms, rr_s):
    """Bazett formülüyle kalp hızına göre düzeltilmiş QT aralığı (ms)."""
    return qt_ms / rr_s ** 0.5


print(round(qtc_bazett(391, 60 / 85), 1))

**Python notu.** `def` bir *fonksiyon* tanımlar. Fonksiyon, adı olan ve tekrar tekrar kullanılabilen bir işlemdir; bu yönüyle bir klinik protokole benzer. `qt_ms` ve `rr_s` fonksiyonun girdileridir, `return` ise sonucu geri verir. Üç tırnak içindeki metin, fonksiyonun kısa açıklamasıdır (docstring). `**` üs alma işlemidir; bu yüzden `rr_s ** 0.5` karekök anlamına gelir.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
rr = np.array([0.81, 0.85, 0.83])   # üç RR aralığı (s)
print(60 / rr)                      # her atımın kalp hızı (atım/dk)
print(rr.mean(), rr.std(ddof=1))

**Python notu.** 0. bölümde `np` adıyla yüklenen NumPy, bir sayı dizisinin bütün elemanları üzerinde tek seferde işlem yapar: `60 / rr`, 60'ı her RR aralığına ayrı ayrı böler. `.mean()` ortalamayı, `.std(ddof=1)` ise paydasında $n-1$ bulunan örneklem standart sapmasını verir.

**Hızlı kontrol.** `troponin.append(4.0)` satırından sonra `troponin[-1]` neyi döndürür?

<details><summary>Cevap</summary><code>4.0</code>: <code>-1</code> indeksi her zaman son elemanı verir ve <code>append()</code>, <code>4.0</code> değerini sona eklemiştir.</details>

**Alıştırma 1.1.** Dakikada 72 atımlık bir kalp hızı için RR aralığını saniye cinsinden hesaplayın ve üç ondalık basamağa yuvarlayın.

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma 1.1", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(60 / 72, 3)
teach.check("Alıştırma 1.1", cevap)

**Alıştırma 1.2.** QT aralığını RR aralığının küp köküne bölen bir `qtc_fridericia(qt_ms, rr_s)` fonksiyonu yazın: $QT_{cF} = QT / RR^{1/3}$. Fonksiyonu haftanın vakası için kullanın (QT 391 ms, kalp hızı dakikada 85 atım) ve sonucu bir ondalık basamağa yuvarlayın.

In [ ]:
def qtc_fridericia(qt_ms, rr_s):
    return None   # None yerine formülü yazın


cevap = qtc_fridericia(391, 60 / 85)
cevap = round(cevap, 1) if cevap is not None else None
teach.check("Alıştırma 1.2", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
def qtc_fridericia(qt_ms, rr_s):
    """Fridericia formülüyle düzeltilmiş QT aralığı (ms)."""
    return qt_ms / rr_s ** (1 / 3)


cevap = round(qtc_fridericia(391, 60 / 85), 1)
teach.check("Alıştırma 1.2", cevap)

**Alıştırma 1.3.** `clinic` listesinde kalp hızı dakikada 80 atımın üzerinde olan hastaları sayın. Bunun için bir `for` döngüsü ve bir `if` ifadesi kullanın.

In [ ]:
count = 0
# döngünüz buraya
cevap = None   # cevabınız
teach.check("Alıştırma 1.3", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
count = 0
for p in clinic:
    if p["hr"] > 80:
        count = count + 1
cevap = count
teach.check("Alıştırma 1.3", cevap)

**pandas ile bir hasta tablosu.** pandas bir tabloyu *DataFrame* içinde tutar: Satırlar hastalar, sütunlar değişkenlerdir. Aşağıdaki küçük tabloda, mini projedeki Cleveland verisinin sütunlarıyla altı hasta yer alır: Yaş (yıl), cinsiyet (1 = erkek, 0 = kadın), total kolesterol (mg/dl) ve koroner arter hastalığı (1 = var). Aynı komutlar 303 ya da 300.000 satırlık bir tabloda da aynı biçimde çalışır.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
patients = pd.DataFrame({
    "age": [63, 67, 37, 41, 56, 62],
    "sex": [1, 1, 1, 0, 1, 0],
    "chol": [233, 286, 250, 204, 236, 268],
    "disease": [0, 1, 0, 0, 0, 1],
})
print(patients)
print("satır ve sütun sayısı:", patients.shape)
print("ortalama kolesterol (mg/dl):", round(patients["chol"].mean(), 1))

**Python notu.** `pd.DataFrame({...})` bir sözlükten tablo oluşturur: Her anahtar bir sütunun adıdır, ardından gelen liste de o sütunun değerlerini tutar. `patients["chol"]` bir sütunu seçer, `.mean()` ise o sütunun ortalamasını alır. `.shape` satır ve sütun sayısını verir.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
older = patients[patients["age"] >= 60]            # koşulu sağlayan satırlar
print(older)
print("60 yaş ve üzerindeki hasta sayısı:", len(older))
print(patients["sex"].value_counts())               # her değerin kaç kez geçtiği
print(patients.groupby("disease")["chol"].mean())   # her gruptaki ortalama kolesterol

**Python notu.** `patients["age"] >= 60` her yaşı 60 ile karşılaştırır ve `True` ile `False` değerlerinden oluşan bir sütun verir; köşeli parantezin içinde kullanıldığında yalnızca `True` olan satırları bırakır. `value_counts()` bir sütundaki her değeri sayar. `groupby("disease")` tabloyu `disease` değerlerine göre gruplara ayırır, `["chol"].mean()` ise her gruptaki ortalama kolesterolü hesaplar.

In [ ]:
# Dersteki çözümlü örnek: Beş kolesterol değeri
chol5 = pd.Series([233, 286, 229, 250, 204])    # Cleveland verisinin ilk beş hastası (mg/dl)
mean = chol5.sum() / len(chol5)
sd = (((chol5 - mean) ** 2).sum() / (len(chol5) - 1)) ** 0.5
print(f"elle ortalama: {mean:.2f}  pandas: {chol5.mean():.2f}")
print(f"elle standart sapma: {sd:.2f}  pandas: {chol5.std():.2f}")
print(f"medyan: {chol5.median():.1f}")

**Python notu.** `pd.Series` tek bir sütundur. İki yol aynı sayıları verir: `.sum() / len(...)` ortalamanın formülüdür, `.mean()` ise aynı hesabın pandas'taki karşılığıdır. `.std()` paydada n − 1 kullanır; bu, ders sayfasındaki örneklem standart sapmasıdır.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
colours = ["tab:red" if d == 1 else "tab:blue" for d in patients["disease"]]
fig, ax = plt.subplots(figsize=(5.5, 3))
ax.bar(range(1, 7), patients["chol"], color=colours)
ax.set_xlabel("hasta")
ax.set_ylabel("total kolesterol (mg/dl)")
ax.set_title("Altı hasta (kırmızı: koroner arter hastalığı)")
plt.show()

**Python notu.** `plt.subplots` bir şekil (`fig`) ve onun eksenlerini (`ax`) oluşturur. `ax.bar` her hasta için bir çubuk çizer. `[... for d in ...]` satırı bir *liste üreteci*dir: Her hasta için bir renk içeren bir liste kurar. Her eksen ölçülen büyüklüğü ve birimini yazar, `plt.show()` ise şekli gösterir.

**Alıştırma 1.4.** `patients` tablosunda koroner arter hastalığı olanların (`disease == 1`) ortalama yaşını hesaplayın ve bir ondalık basamağa yuvarlayın.

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma 1.4", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(patients.loc[patients["disease"] == 1, "age"].mean(), 1)
teach.check("Alıştırma 1.4", cevap)

**«Durak 1 · Python temelleri» tamamlandı.** Sırada hemen aşağıdaki «Durak 1 · Prompt ile uygulama» bölümü var. Durağı burada bırakacaksanız [ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#last-two-hours-working-with-ecg-data) ve okumaya «Son 2 saat: EKG verisi ile çalışma» bölümünden devam edin.

### Durak 1 · Prompt ile uygulama

Bu bölümün üç promptu, Cleveland verisini ve sanal kliniği bir ortalama, bir yüzde ve bir sayıyla özetler.

Her promptta dört adım vardır. Promptu kopyalayıp YZ asistanınıza gönderin. Altındaki veri hücresini çalıştırın; bu hücre, promptun tarif ettiği "veri" tablosunu hazırlar. Asistanın kodunu boş hücreye yapıştırıp çalıştırın. Ardından kontrol hücresini çalıştırın; hücre *doğru* ya da *henüz değil* yazar. Asistanın kodu kontrolden geçmezse *Hazır kod* hücresini çalıştırıp devam edin.

**Promptlarla çalışmak.** Python kodu yazabilen bir YZ asistanı yeterlidir. Google Colab'ın içinde Gemini bulunur ve pencerenin üstündeki Gemini düğmesiyle açılır; başka bir tarayıcı sekmesinde açılan ChatGPT, Claude, Copilot ya da Gemini de aynı işi görür. Haftanın bütün promptları [prompt kütüphanesinde](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/PROMPTLAR.md) de yer alır; GitHub her promptun yanında bir kopyalama düğmesi gösterir.

**Her promptta geçen iki ad: "veri" ve "cevap".** YZ asistanı çoğu zaman Jupyter Notebook'u göremez; bu yüzden promptlar verileri sözcüklerle tarif eder. Her promptta veri hücresi, o promptun verilerini "veri" adlı bir tabloya koyar; prompt da bu tablonun satırlarını ve sütunlarını anlatır. Prompt ayrıca asistandan sonucu "cevap" adıyla saklamasını ister; kontrol hücresi sonucu bu addan okur. Eğitimin promptlarında koddan yalnızca bu iki ad geçer.

**Onarım promptu** (çalışan ama kontrolden geçmeyen kod için):

> Aşağıdaki kod şu görev için yazıldı: [promptun Görev satırını yapıştırın]. Kod çalışıyor, ancak dersin kontrolü "henüz değil" yazıyor. Hatayı bul, iki cümleyle açıkla ve düzeltilmiş kodu ver. Sonuç yine "cevap" adıyla saklanmalı.  
>
> [kodu buraya yapıştırın]  

**Hata mesajı için prompt** (hata vererek duran kod için):

> Aşağıdaki kod bir Jupyter Notebook'ta çalıştırıldığında altta verilen hata mesajını veriyor. Hatanın nedenini sade bir dille açıkla ve düzeltilmiş kodu ver.  
>
> [kodu buraya yapıştırın]  
>
> [hata mesajını buraya yapıştırın]  

**Hiç değişmeyen kural.** Promptlarda yalnızca eğitimin sentetik ya da açık verileri bulunur. Bir YZ asistanına gerçek hasta verisi asla yapıştırılmaz.

In [ ]:
#@title Promptların hazırlığı: Bu hücreyi bir kez çalıştırın (verileri yükler; 0. bölüm bu oturumda çalışmadıysa önce onu çalıştırır)
VERI_KAYNAGI = "web"   #@param ["kütüphane", "web", "bilgisayar"]
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
from vclinic.datasets import load_cleveland
heart = load_cleveland(source=VERI_KAYNAGI)
clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
print("1. durağın verileri hazır.")

#### Prompt 1.1: Hastalığı olanların ortalama yaşı

**Amaç.** Bu promptla koroner arter hastalığı olan Cleveland hastalarının ortalama yaşı hesaplanır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Cleveland Clinic'te koroner anjiyografi yapılmış 303 hastada, koroner arter hastalığı olanların ortalama yaşını öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var. Her satır bir hastadır: "yaş" sütunu hastanın yaşını yıl olarak verir. "KAH" sütununda, anjiyografide en az bir büyük koroner arterinin çapı yüzde 50'den fazla daralmış hastalar için "var", diğer hastalar için "yok" yazar.  
> Görev: Koroner arter hastalığı olan hastaları seç ve ortalama yaşlarını hesapla. Kaç hasta seçildiğini de yazdır.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Ortalama yaşı yıl olarak, bir ondalık basamakla yazdır ve "cevap" adıyla sakla.  
> Test: Karşılaştırma için bütün hastaların ortalama yaşını da yazdır; bu değer 54,4 yıl olmalı. Hastalığı olanların ortalaması bundan farklı çıkmalı.  

In [ ]:
#@title Prompt 1.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["clinic", "heart", "load_cleveland"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    from vclinic.datasets import load_cleveland
    heart = load_cleveland(source=VERI_KAYNAGI)
    clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
veri = pd.DataFrame({"yaş": heart["age"],
                     "KAH": heart["disease"].map({1: "var", 0: "yok"})})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Prompt 1.1 kontrolü
teach.check_step("Prompt 1.1", "cevap", globals())

In [ ]:
#@title Prompt 1.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["clinic", "heart", "load_cleveland"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    from vclinic.datasets import load_cleveland
    heart = load_cleveland(source=VERI_KAYNAGI)
    clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
veri = pd.DataFrame({"yaş": heart["age"],
                     "KAH": heart["disease"].map({1: "var", 0: "yok"})})
hastalar = veri[veri["KAH"] == "var"]
print("Hastalığı olan hasta sayısı:", len(hastalar))
print("Bütün hastaların ortalama yaşı:", round(veri["yaş"].mean(), 1))
cevap = round(hastalar["yaş"].mean(), 1)
print("Hastalığı olanların ortalama yaşı:", cevap)
teach.check_step("Prompt 1.1", "cevap", globals())

**Sonuç ne gösteriyor?** Hastalığı olanlar, grubun tamamından ortalama iki yaş kadar büyüktür. Bir ortalama tek tek hastalar hakkında pek az bilgi verir; bu yüzden ders sayfasındaki tablolar dağılımı da göstermektedir.

#### Prompt 1.2: Kadınların oranı

**Amaç.** Bu promptla Cleveland hastaları arasında kadınların oranı bulunur.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Cleveland Clinic'te koroner anjiyografi yapılmış 303 hastanın yüzde kaçının kadın olduğunu öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var. Her satır bir hastadır: "yaş" sütunu hastanın yaşını yıl olarak verir; "cinsiyet" sütununda "kadın" ya da "erkek" yazar.  
> Görev: Kadın hastaların sayısını bütün hastaların sayısına böl ve sonucu yüzde olarak ifade et. Erkeklerin yüzdesini de aynı yolla hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Kadınların yüzdesini bir ondalık basamakla yazdır ve "cevap" adıyla sakla. Erkeklerin yüzdesini de yazdır.  
> Test: Kadınların ve erkeklerin yüzdelerinin toplamı 100 olmalı.  

In [ ]:
#@title Prompt 1.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["clinic", "heart", "load_cleveland"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    from vclinic.datasets import load_cleveland
    heart = load_cleveland(source=VERI_KAYNAGI)
    clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
veri = pd.DataFrame({"yaş": heart["age"],
                     "cinsiyet": heart["sex"].map({1: "erkek", 0: "kadın"})})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Prompt 1.2 kontrolü
teach.check_step("Prompt 1.2", "cevap", globals())

In [ ]:
#@title Prompt 1.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["clinic", "heart", "load_cleveland"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    from vclinic.datasets import load_cleveland
    heart = load_cleveland(source=VERI_KAYNAGI)
    clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
veri = pd.DataFrame({"yaş": heart["age"],
                     "cinsiyet": heart["sex"].map({1: "erkek", 0: "kadın"})})
kadin = 100 * (veri["cinsiyet"] == "kadın").mean()
erkek = 100 * (veri["cinsiyet"] == "erkek").mean()
cevap = round(kadin, 1)
print("Kadın (%):", cevap, " Erkek (%):", round(erkek, 1))
teach.check_step("Prompt 1.2", "cevap", globals())

**Sonuç ne gösteriyor?** Cleveland verisindeki hastaların yaklaşık üçte ikisi erkektir. Böyle bir veriyle eğitilen bir model kadınlar hakkında daha az şey öğrenir; 9. hafta bu durumu bir yanlılık sorunu olarak ele almaktadır.

#### Prompt 1.3: Sanal klinikte HFrEF hastaları

**Amaç.** Bu promptla sanal klinikteki düşük ejeksiyon fraksiyonlu kalp yetersizliği (HFrEF) hastaları sayılır ve her tanının ejeksiyon fraksiyonu tek bir şekilde gösterilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Sanal kliniğin 500 sentetik hastasında düşük ejeksiyon fraksiyonlu kalp yetersizliği (HFrEF) olanları saymak ve ejeksiyon fraksiyonunun tanılara göre nasıl değiştiğini görmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var. Her satır bir hastadır: "hasta" sütunu hastanın numarasını, "ana tanı" sütunu ana tanıyı, "LVEF" sütunu da sol ventrikül ejeksiyon fraksiyonunu yüzde olarak verir. Ana tanı şu değerlerden biridir: "yok" (tanı yok), "CCS" (kronik koroner sendrom), "ACS" (akut koroner sendrom), "HFrEF" (düşük ejeksiyon fraksiyonlu kalp yetersizliği) ve "HFpEF" (korunmuş ejeksiyon fraksiyonlu kalp yetersizliği).  
> Görev: Ana tanısı HFrEF olan hastaları say. Ardından her ana tanı için ejeksiyon fraksiyonunun kutu grafiğini çiz; tanılar yatay eksende, yüzde cinsinden ejeksiyon fraksiyonu dikey eksende yer alsın.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: HFrEF hastalarının sayısını yazdır ve "cevap" adıyla sakla. Grafiği de göster.  
> Test: Her tanıdaki hasta sayısını da yazdır; beş sayının toplamı 500 olmalı. Hiçbir HFrEF hastasının ejeksiyon fraksiyonu yüzde 40'ı geçmemeli.  

In [ ]:
#@title Prompt 1.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["clinic", "heart", "load_cleveland"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    from vclinic.datasets import load_cleveland
    heart = load_cleveland(source=VERI_KAYNAGI)
    clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
veri = pd.DataFrame({"hasta": clinic["patient_id"],
                     "ana tanı": clinic["main_diagnosis"].replace({"none": "yok"}),
                     "LVEF": clinic["lvef"]})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Prompt 1.3 kontrolü
teach.check_step("Prompt 1.3", "cevap", globals())

In [ ]:
#@title Prompt 1.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["clinic", "heart", "load_cleveland"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    from vclinic.datasets import load_cleveland
    heart = load_cleveland(source=VERI_KAYNAGI)
    clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
veri = pd.DataFrame({"hasta": clinic["patient_id"],
                     "ana tanı": clinic["main_diagnosis"].replace({"none": "yok"}),
                     "LVEF": clinic["lvef"]})
print(veri["ana tanı"].value_counts())
cevap = int((veri["ana tanı"] == "HFrEF").sum())
sira = ["yok", "CCS", "ACS", "HFpEF", "HFrEF"]
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.boxplot([veri.loc[veri["ana tanı"] == d, "LVEF"] for d in sira])
ax.set_xticks(range(1, 6), sira)
ax.set_xlabel("Ana tanı")
ax.set_ylabel("LVEF (%)")
plt.show()
print("HFrEF hastaları:", cevap)
teach.check_step("Prompt 1.3", "cevap", globals())

**Sonuç ne gösteriyor?** Kayıtta her HFrEF hastasının ejeksiyon fraksiyonu, Avrupa Kardiyoloji Derneği tanımının gerektirdiği gibi yüzde 40 ya da daha düşüktür; her HFpEF hastasınınki ise yüzde 50 ya da daha yüksektir. Sanal kliniği sade tutmak için hafif azalmış ejeksiyon fraksiyonlu kalp yetersizliği (HFmrEF, yüzde 41-49) dışarıda bırakılmıştır.

**«Durak 1 · Prompt ile uygulama» tamamlandı.** Sırada hemen aşağıdaki «Durak 1 · Hazır uygulama» bölümü var. Durağı burada bırakacaksanız [ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#last-two-hours-working-with-ecg-data) ve okumaya «Son 2 saat: EKG verisi ile çalışma» bölümünden devam edin.

### Durak 1 · Hazır uygulama

Bu bölüm mini projenin tamamını içerir: Cleveland verisi üç kaynaktan biriyle yüklenir, denetlenir, hastalık durumuna göre özetlenir ve çizilir; sanal klinik de onun yanında açılır. Son hücre üç promptun sonuçlarını hesaplar.

Hücreleri yukarıdan aşağıya çalıştırın. *Python notu* hücreleri kodu açıklar, *Gözlemle ve düşün* hücreleri ise sonuçlar üzerine sorular sorar.

In [ ]:
#@title Hazır uygulamanın hazırlığı: Bu hücreyi bir kez çalıştırın (0. bölüm ve uygulamanın ihtiyaç duyduğu önceki kodlar bu oturumda çalışmadıysa onları çalıştırır)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    _catch_up(1, "0. bölüm çalıştı.")
print("1. durağın hazır uygulaması hazır.")

#### Mini proje: Kaggle Heart Disease veri setini pandas ile yükleme, istatistiksel özet ve görselleştirme

*Ders: [Mini proje: Kaggle Heart Disease veri setini yükleme, istatistiksel özet ve görselleştirme](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#mini-project-loading-the-kaggle-heart-disease-data-set-statistical-summary-and-visualisation)*

Kaggle'da paylaşılan Heart Disease veri seti, UCI Heart Disease koleksiyonunun Cleveland verisinden alınmıştır ve koroner anjiyografi yapılmış 303 hastanın 14 değişkenini içermektedir. `num` sütunu hastalığı 0'dan (yüzde 50'den fazla daralma yok) 4'e kadar derecelendirir. Eğitim, UCI deposundaki özgün dosyayı birlikte getirdiği için Kaggle hesabı gerekmez. Bu bölüm haftanın ders içi mini projesidir: Veriyi pandas ile yükleyin, işleyin, istatistiklerle özetleyin ve Matplotlib ile görselleştirin.

**Formüllerden koda.** $\bar{x}$ ortalaması `df["chol"].mean()` ile hesaplanır. Paydasında $n-1$ bulunan $s$ örneklem standart sapması `df["chol"].std()` ile bulunur. Medyan `.median()`, çeyrekler arası açıklık ise `.quantile(0.75) - .quantile(0.25)` ile hesaplanır.

In [ ]:
# Cleveland verisini üç kaynaktan biriyle yükleme
VERI_KAYNAGI = "web"   #@param ["kütüphane", "web", "bilgisayar"]

COLUMNS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
           "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]
FILE = "processed.cleveland.data"
URL = vclinic.COURSE_RAW + "data/uci-heart-disease/" + FILE   # UCI dosyasının ders deposundaki kopyası

if VERI_KAYNAGI == "kütüphane":
    # UCI Machine Learning Repository'nin Python kütüphanesi ucimlrepo, veri setini numarasıyla indirir
    vclinic.install("ucimlrepo")
    from ucimlrepo import fetch_ucirepo
    uci = fetch_ucirepo(id=45)
    heart = pd.concat([uci.data.features, uci.data.targets], axis=1)[COLUMNS]
elif VERI_KAYNAGI == "web":
    # pandas dosyayı doğrudan web adresinden okur
    heart = pd.read_csv(URL, header=None, names=COLUMNS, na_values="?")
else:
    # bilgisayardaki dosya: Dosyayı önce bu hücrenin altındaki nottaki bağlantıyla indirin
    path = vclinic.local_file(FILE)   # Jupyter bu Jupyter Notebook'un klasörüne bakar; Colab bir yükleme penceresi açar
    heart = pd.read_csv(path, header=None, names=COLUMNS, na_values="?")

heart["disease"] = (heart["num"] > 0).astype(int)   # büyük bir damarda yüzde 50'den fazla daralma varsa 1
print("kaynak:", VERI_KAYNAGI, "| satır ve sütun sayısı:", heart.shape)
heart.head()

**Python notu.** Aynı tablo üç kaynaktan gelebilir. `fetch_ucirepo(id=45)` UCI deposundan 45 numaralı veri setini ister ve değişkenleri ile sonucu iki tablo olarak döndürür; `pd.concat` bu iki tabloyu yan yana koyar. `pd.read_csv` bir dosyayı bilgisayardan okuduğu gibi bir web adresinden de okur: `header=None` dosyada sütun adları satırı olmadığını belirtir, `names=COLUMNS` adları verir, `na_values="?"` ise dosyadaki soru işaretlerini eksik değere çevirir.

**Veriyi kendi bilgisayarınızdan okumak.** `"bilgisayar"` kaynağı için önce [uci_heart_disease_cleveland.zip](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/raw/main/data/downloads/uci_heart_disease_cleveland.zip) dosyasını indirin. Zip dosyasını açın ve içindeki `processed.cleveland.data` dosyasını bu Jupyter Notebook'la aynı klasöre koyun. Ardından kaynak olarak `"bilgisayar"` seçin ve veri hücresini yeniden çalıştırın. Google Colab'da hücre, dosyayı bilgisayarınızdan seçmeniz için bir düğme gösterir. Zip dosyasında verinin atıf bilgisi ve lisansı da yer alır.

**Python notu.** `pd` adıyla yüklenen pandas, bir tabloyu *DataFrame* içinde tutar: Satırlar hastalar, sütunlar değişkenlerdir. Yukarıdaki hücre dosyayı okur ve sütunlara ad verir. `.shape` satır ve sütun sayısını, `.head()` ise ilk beş satırı gösterir.

| Sütun | Anlamı |
|---|---|
| `age`, `sex` | Yıl olarak yaş; cinsiyet (1 = erkek, 0 = kadın) |
| `cp` | Göğüs ağrısı tipi (1 tipik anjina, 2 atipik anjina, 3 anjina dışı ağrı, 4 asemptomatik) |
| `trestbps` | İstirahat sistolik kan basıncı (mmHg) |
| `chol` | Serum kolesterolü (mg/dL) |
| `fbs` | Açlık kan şekerinin 120 mg/dL'nin üzerinde olması (1 = evet) |
| `restecg` | İstirahat EKG'si (0 normal, 1 ST-T anormalliği, 2 sol ventrikül hipertrofisi) |
| `thalach` | Efor testinde ulaşılan en yüksek kalp hızı (atım/dk) |
| `exang` | Eforla ortaya çıkan anjina (1 = evet) |
| `oldpeak` | İstirahate göre eforla oluşan ST çökmesi (mm) |
| `slope` | Doruk eforda ST segmentinin eğimi (1 yukarı eğimli, 2 düz, 3 aşağı eğimli) |
| `ca` | Floroskopiyle boyanan ana damar sayısı (0 ile 3 arası) |
| `thal` | Talyum sintigrafisi (3 normal, 6 sabit defekt, 7 geri dönüşlü defekt) |
| `num`, `disease` | Anjiyografik hastalık derecesi 0 ile 4 arası; `num` > 0 ise `disease` = 1 |

In [ ]:
# Hastalık durumuna göre temel değişkenlerin özeti
cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]
print(heart["disease"].value_counts())
heart.groupby("disease")[cols].agg(["mean", "std"]).round(1)

**Python notu.** `value_counts()` bir sütundaki her değerin kaç kez geçtiğini sayar. `groupby("disease")` tabloyu hastalığı olan ve olmayan hastalar olarak ikiye ayırır; `.agg(["mean", "std"])` ise her grup ve her sütun için bu iki istatistiği hesaplar.

In [ ]:
# Tabloyu işlemek: Eksik değerler, bir seçim, yeni bir sütun ve sıralı bir görünüm
print(heart.isna().sum()[heart.isna().sum() > 0])        # sütunlara göre eksik değerler
older = heart[heart["age"] >= 60]                          # koşulu sağlayan satırlar
print("60 yaş ve üzerindeki hasta sayısı:", len(older))
heart["chol_mmol"] = (heart["chol"] / 38.67).round(2)      # yeni bir sütun: mmol/L cinsinden kolesterol
heart.sort_values("thalach")[["age", "sex", "thalach", "disease"]].head()

**Python notu.** `heart.isna().sum()` her sütundaki eksik değerleri sayar. `heart[heart["age"] >= 60]` koşulun doğru olduğu satırları tutar. Yeni bir sütun atamayla oluşturulur; burada total kolesterol 38,67'ye bölünerek mg/dL'den mmol/L'ye çevrilmiştir. `sort_values("thalach")` satırları en yüksek kalp hızına göre sıralar; çift köşeli parantez içindeki bir ad listesi de sütunları seçer.

**Matplotlib ile ilk kardiyolojik grafik.** *Ders: [Matplotlib ile ilk kardiyolojik grafik](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#a-first-cardiological-plot-with-matplotlib)*

`plt.subplots(1, 2)` yan yana iki eksen takımı olan bir şekil oluşturur. `hist` her gruptaki yaşın histogramını çizer, `boxplot` ise iki grubun en yüksek kalp hızını karşılaştırır. Her eksen ölçülen büyüklüğü ve birimini yazar.

In [ ]:
# Kaydın ilk iki şekli
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for value, color, label in [(0, "#1F5F8B", "hastalık yok"),
                            (1, "#9B1B1B", "hastalık var")]:
    sub = heart[heart["disease"] == value]
    axes[0].hist(sub["age"], bins=15, alpha=0.6, color=color, label=label)
axes[0].set_xlabel("Yaş (yıl)")
axes[0].set_ylabel("Hasta sayısı")
axes[0].legend()
heart.boxplot(column="thalach", by="disease", ax=axes[1])
axes[1].set_title("")
axes[1].set_xlabel("Hastalık (0 yok, 1 var)")
axes[1].set_ylabel("En yüksek kalp hızı (atım/dk)")
fig.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
# Aynı karşılaştırma Matplotlib üzerine kurulu Seaborn ile
try:
    import seaborn as sns
    fig, ax = plt.subplots(figsize=(5.5, 4))
    sns.boxplot(data=heart, x="disease", y="thalach", color="#C9A0A0", ax=ax)
    ax.set_xlabel("Hastalık (0 yok, 1 var)")
    ax.set_ylabel("En yüksek kalp hızı (atım/dk)")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Seaborn kurulu değil; yukarıdaki Matplotlib şekli aynı karşılaştırmayı göstermektedir.")

**Gözlemle ve düşün.** Özet tablosunda iki grup en çok hangi değişkende farklılaşıyor? Farkın yönü klinik açıdan akla yatkın mı? Kadın hasta sayısının az olması, sonuçların ne kadar genellenebileceği konusunda ne düşündürüyor?

**İleri düzey (isteğe bağlı).** İki sütunda eksik değer vardır. Bunları `heart.isna().sum()` ile sayın, ardından göğüs ağrısı tipine göre hastalık oranını `pd.crosstab(heart["cp"], heart["disease"], normalize="index")` ile karşılaştırın.

In [ ]:
# İleri düzey (isteğe bağlı)
print(heart.isna().sum()[heart.isna().sum() > 0])
pd.crosstab(heart["cp"], heart["disease"], normalize="index").round(2)

#### Sanal klinik açılıyor

*Ders: [Sanal kardiyoloji kliniği: Eğitimin projesi](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#the-virtual-cardiology-clinic-the-project-of-the-course)*

`vclinic.make_cohort()` sanal kardiyoloji kliniğinin hasta kaydını oluşturur. Her hastanın demografik verileri, risk faktörleri, laboratuvar değerleri, sol ventrikül ejeksiyon fraksiyonu (LVEF), ana tanısı ve istirahat 12 derivasyonlu elektrokardiyogramının (EKG) parametreleri vardır. Her hasta, eğitimin bütün haftalarında aynı değerleri taşır. Veriler sentetiktir: Genel klinik örüntüleri izlerler ama gerçek bir kişiyi tanımlamazlar.

In [ ]:
clinic = vclinic.make_cohort(n=500, seed=2026)
print(clinic.shape)
clinic.head()

**Python notu.** `vclinic.make_cohort(n=500, seed=2026)` satırı, ders paketindeki bir fonksiyonu iki *anahtar sözcüklü argümanla* çağırır. Seed değeri rastgele sayıları sabitler; böylece her çalıştırmada aynı 500 hasta elde edilir.

In [ ]:
# Kliniğin ana tanıları ve ritmi
print(clinic["main_diagnosis"].value_counts())
print(clinic["ecg_rhythm"].value_counts())
clinic[["age", "bmi", "sbp", "ldl", "hdl", "egfr", "lvef", "ntprobnp",
        "ecg_hr", "ecg_pr", "ecg_qrs", "ecg_qt"]].describe().round(1)

Tanı kodları uluslararası kısaltmalardır: `CCS` kronik koroner sendromu (KKS), `ACS` akut koroner sendromu (AKS), `HFrEF` düşük ejeksiyon fraksiyonlu kalp yetersizliğini, `HFpEF` korunmuş ejeksiyon fraksiyonlu kalp yetersizliğini, `none` ise yapısal kalp hastalığı olmadığını gösterir. Ritim sütunundaki `AF` atriyal fibrilasyonu gösterir. Veri sözlüğünün tamamı DATASETS.tr.md dosyasındadır.

In [ ]:
# Sanal klinik ve Cleveland verisi yan yana
compare = pd.DataFrame({
    "Cleveland (gerçek)": [
        heart["age"].mean(), 100 * heart["sex"].mean(),
        heart["trestbps"].mean(), heart["chol"].mean(),
        100 * heart["disease"].mean()],
    "Sanal klinik (sentetik)": [
        clinic["age"].mean(), 100 * (clinic["sex"] == "M").mean(),
        clinic["sbp"].mean(), clinic["total_chol"].mean(),
        100 * clinic["cad"].mean()],
}, index=["Ortalama yaş (yıl)", "Erkek (%)",
          "Sistolik basınç (mmHg)",
          "Total kolesterol (mg/dL)",
          "Koroner hastalık (%)"])
compare.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.hist(heart["age"], bins=20, density=True, alpha=0.55, color="#1F5F8B",
        label="Cleveland (gerçek)")
ax.hist(clinic["age"], bins=20, density=True, alpha=0.55, color="#9B1B1B",
        label="Sanal klinik (sentetik)")
ax.set_xlabel("Yaş (yıl)")
ax.set_ylabel("Yoğunluk")
ax.legend()
plt.tight_layout()
plt.show()

**Gözlemle ve düşün.** İki kohort tabloda birbirine benziyor. Yine de sentetik bir kayıt, bu kayıtla eğitilecek bir model açısından önemli olabilecek hangi yönlerden gerçek bir kayıttan ayrılabilir? Nadir durumları, değişkenler arasındaki ilişkileri ve ölçüm gürültüsünü düşünün.

**Promptların sonuçları.** Aşağıdaki hücre 1. durağın promptlarının sonuçlarını bu uygulamanın koduyla hesaplar ve promptların kontrol hücreleri gibi denetler.

In [ ]:
# Bu durağın promptlarının sonuçları, Python ile
mean_age = round(heart.loc[heart["disease"] == 1, "age"].mean(), 1)
print("Koroner arter hastalığı olanların ortalama yaşı:", mean_age)
teach.check("Prompt 1.1", mean_age)
women = round(100 * (heart["sex"] == 0).mean(), 1)
print("Kadınların yüzdesi:", women)
teach.check("Prompt 1.2", women)
hfref = int((clinic["main_diagnosis"] == "HFrEF").sum())
print("Sanal klinikteki HFrEF hastası sayısı:", hfref)
teach.check("Prompt 1.3", hfref)

**1. durak tamamlandı.** [Ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#last-two-hours-working-with-ecg-data) ve okumaya «Son 2 saat: EKG verisi ile çalışma» bölümünden devam edin. Bir sonraki uygulama durağında ders sayfası sizi yine bu Jupyter Notebook'a getirir.

## Durak 2: Veri olarak EKG

*Ders sayfasında: [2. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#stop-2)*

Dijital bir EKG de bir sayı tablosudur: Her derivasyon bir sütunda, her örnek bir satırda yer alır. Bu durakta haftanın vakası olan VC-0317 numaralı hastanın EKG'sinden D2 derivasyonu çizilir ve ölçülür; başka bir hastanın QT aralığı da iki formülle kalp hızına göre düzeltilir.

**Durağın sonunda:** 2,5 saniyede 1250 örnek içeren bir D2 şeridini, R tepelerinden hesaplanan kalp hızını (84,9 atım/dk) ve Fridericia formülüyle düzeltilmiş QT aralığını (403,9 ms) elde edeceksiniz.

Durak üç bölümden oluşur: **Python temelleri**, **Prompt ile uygulama** ve **Hazır uygulama**. Bölümleri bu sırayla izleyin. Python biliyorsanız promptlarla, yalnızca sonucu görmek istiyorsanız hazır uygulamayla başlayabilirsiniz. Her bölümün ilk hücresi, bölümün ihtiyaç duyduğu her şeyi hazırlar.

### Durak 2 · Python temelleri

*Ders sayfasında: [Veri olarak EKG](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#the-ecg-as-data)*

Bir EKG derivasyonu, düzenli aralıklarla ölçülmüş gerilimlerin bir listesidir. Bu bölümün örnekleri gerçek bir EKG yerine basit bir dalga kullanır; Python'un bir sinyalle nasıl çalıştığını görmek için bu kadarı yeterlidir. Bölümde zaman ve örnekler, dilimler, koşullar, bir matris, bir döngü ve bir şekil ele alınır.

Hücreleri sırayla çalıştırın. Bir örneğin altındaki *Python notu* yeni yapıyı açıklar. Alıştırmalar hemen *doğru* ya da *henüz değil* yazar; örnek çözümler katlanmış hücrelerdedir.

In [ ]:
#@title Python temellerinin hazırlığı: Bu hücreyi bir kez çalıştırın (0. bölüm bu oturumda çalışmadıysa onu çalıştırır)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
print("2. durağın Python temelleri hazır.")

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
fs_demo = 500                             # saniyedeki örnek sayısı (Hz)
t = np.arange(0, 2, 1 / fs_demo)          # iki saniye; her 2 ms'de bir zaman noktası
wave = np.sin(2 * np.pi * 1.25 * t)       # saniyede 1,25 kez tekrarlanan düzgün bir dalga
print("örnek sayısı:", len(t))
print("ilk beş zaman noktası (s):", t[:5])
print("zaman adımı (ms):", 1000 / fs_demo)

**Python notu.** `np.arange(0, 2, 1 / fs_demo)` 0'dan 2'ye kadar, 0,002'lik adımlarla sayılar üretir; bitiş değeri olan 2 bu sayılara dâhil değildir. `np.sin(...)` dizinin tamamı üzerinde tek seferde çalışır ve her zaman noktası için bir değer verir. `len(t)` eleman sayısını sayar, `t[:5]` ise ilk beş elemanı alır.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
i = int(0.5 * fs_demo)                                  # 0,5. saniyedeki örnek
print("örnek", i, "zamanı", t[i], "s:", round(wave[i], 3))
window = wave[int(0.2 * fs_demo):int(0.6 * fs_demo)]    # 0,2 ile 0,6 saniye arasındaki örnekler
print("penceredeki örnek sayısı:", len(window))

**Python notu.** Python saymaya 0'dan başlar; bu yüzden ilk örnek `wave[0]`'dır. Bir zaman, örnekleme frekansıyla çarpıldığında örnek numarasına dönüşür: 0,5 s × 500 Hz, 250 numaralı örneği verir. `a:b` dilimi `a`'dan `b`'ye kadar olan elemanları alır, `b`'nin kendisini almaz; bu yüzden 0,4 saniyelik pencerede 200 örnek bulunur.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
late = t >= 1.0                                         # 1. saniyeden sonraki her örnek için True
print("1. saniyeden sonraki örnek sayısı:", late.sum())
top = np.argmax(wave[late])                             # o kısımdaki en büyük değerin yeri
print("1. saniyeden sonraki en yüksek nokta (s):", round(t[late][top], 3))

**Python notu.** `t >= 1.0` her zamanı 1,0 ile karşılaştırır ve `True` ile `False` değerlerinden oluşan bir dizi, yani bir *maske* verir. `wave[late]` yalnızca maskenin `True` olduğu değerleri bırakır; `late.sum()` ise bunları sayar, çünkü `True` 1 olarak sayılır. `np.argmax` en büyük değerin konumunu verir.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
directions = np.array([[1.0, 0.0, 0.0],                 # uzayda üç yön; her satırda bir tane
                       [0.0, 1.0, 0.0],
                       [0.5, 0.866, 0.0]])
v = np.array([0.6, 0.8, -0.2])                          # bir vektör (mV)
print(directions @ v)                                   # her satır için bir sayı
print("üçüncü satır, elle:", round(0.5 * 0.6 + 0.866 * 0.8 + 0.0 * -0.2, 4))

**Python notu.** `@` bir matrisi bir vektörle çarpar: Her satır vektörle eleman eleman çarpılır ve çarpımlar toplanır. Sonuç, vektörün her yöndeki izdüşümüdür. Ders sayfası EKG'nin on iki derivasyonunu aynı yolla, tek bir kalp vektöründen ve on iki derivasyon yönünden kurar.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
values = [0.05, 0.3, 0.9, 1.4, 1.1, 0.7, 0.35, 0.1]     # tepesi çevresinde bir dalga
k = values.index(max(values))                           # tepeden başla
while k + 1 < len(values) and values[k + 1] > 0.3:
    k += 1                                              # değer 0,3'ün üzerinde kaldıkça sağa ilerle
print("dalganın 0,3'ün üzerinde kaldığı son konum:", k)

**Python notu.** `while`, koşulu doğru kaldığı sürece girintili satırlarını tekrarlar. Burada döngü tepeden başlar ve bir sonraki değer 0,3 ya da altına düşene kadar sağa ilerler. Ders sayfasındaki ölçüm, QRS kompleksinin ve T dalgasının sonunu, örnek örnek ilerleyen bu tür aramalarla bulur. `k += 1`, `k = k + 1` ifadesinin kısa yazılışıdır.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
def qtc_category(qtc_ms, sex):
    limit = 460 if sex == "F" else 450                  # normalin üst sınırı (ms); "F" kadın demektir
    if qtc_ms > 500:
        return "belirgin uzun"
    elif qtc_ms > limit:
        return "uzun"
    else:
        return "normal"


for qtc in (430, 455, 470, 510):
    print(qtc, "ms:", qtc_category(qtc, "F"))

**Python notu.** `elif`, *değilse eğer* anlamına gelir: Python koşulları yukarıdan aşağıya sınar ve yalnızca koşulu doğru olan ilk bloğu çalıştırır. `limit = 460 if sex == "F" else 450` satırı iki değerden birini tek satırda seçer. Sınırlar, 2009 önerilerinin kadınlar ve erkekler için verdiği değerlerdir.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
def describe(values):
    """Bir sayı listesinin ortalaması ve aralığı."""
    arr = np.asarray(values, dtype=float)
    return arr.mean(), arr.max() - arr.min()


mean_qrs, range_qrs = describe([92, 96, 88, 94, 90])    # beş atımın QRS süreleri (ms)
print("ortalama:", mean_qrs, "ms | aralık:", range_qrs, "ms")

**Python notu.** Bir fonksiyon birden çok değeri aynı anda döndürebilir: `return a, b` bir çift döndürür, `mean_qrs, range_qrs = ...` ise bu çifti iki ada ayırır. `def` satırının altında üçlü tırnak içindeki metin, fonksiyonun ne yaptığını kısaca anlatan açıklamadır; buna *docstring* denir.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
fig, ax = plt.subplots(figsize=(8, 2.6))
ax.plot(t, wave, lw=1)
ax.axvline(1.0, color="tab:red", ls="--", lw=1)         # 1. saniyede dikey bir çizgi
ax.set_xlabel("zaman (s)")
ax.set_ylabel("genlik")
ax.grid(alpha=0.3)
plt.show()

**Python notu.** `ax.plot(x, y)` (x, y) noktalarını bir çizgiyle birleştirir; bir sinyalde x zaman, y ise değerdir. `axvline` dikey bir çizgi çizer, örneğin bir ölçüm noktasında; `grid` ise EKG kâğıdındakilere benzeyen açık renkli kılavuz çizgileri ekler.

**Alıştırma 2.1.** EKG kâğıdında bir büyük kare 0,2 saniyeye karşılık gelir. 500 Hz'de bir büyük kare kaç örnek içerir?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma 2.1", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(0.2 * 500)
teach.check("Alıştırma 2.1", cevap)

**Alıştırma 2.2.** Kalp vektörü $\mathbf{H} = (0{,}6;\ 0{,}8;\ -0{,}2)$ mV, aVL derivasyonunun yönü ise $(0{,}75;\ -0{,}433;\ 0)$ olsun. aVL derivasyonundaki gerilimi `@` ile hesaplayın ve üç ondalık basamağa yuvarlayın.

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma 2.2", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float(np.array([0.75, -0.433, 0.0]) @ np.array([0.6, 0.8, -0.2])), 3)
teach.check("Alıştırma 2.2", cevap)

**«Durak 2 · Python temelleri» tamamlandı.** Sırada hemen aşağıdaki «Durak 2 · Prompt ile uygulama» bölümü var. Durağı burada bırakacaksanız [ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#real-data-physionet-records) ve okumaya «PhysioNet ve MIT-BIH veri setlerine erişim» bölümünden devam edin.

### Durak 2 · Prompt ile uygulama

Bu bölümün üç promptu haftanın vakasının EKG'siyle çalışır: Bir D2 şeridi, R tepelerinden kalp hızı ve Fridericia formülüyle QTc. Onarım promptu ve hata mesajı için prompt, [prompt kütüphanesinde](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/PROMPTLAR.md) ve 1. durağın promptlarında yer alır.

Her promptta dört adım vardır. Promptu kopyalayıp YZ asistanınıza gönderin. Altındaki veri hücresini çalıştırın; bu hücre, promptun tarif ettiği "veri" tablosunu hazırlar. Asistanın kodunu boş hücreye yapıştırıp çalıştırın. Ardından kontrol hücresini çalıştırın; hücre *doğru* ya da *henüz değil* yazar. Asistanın kodu kontrolden geçmezse *Hazır kod* hücresini çalıştırıp devam edin.

In [ ]:
#@title Promptların hazırlığı: Bu hücreyi bir kez çalıştırın (verileri yükler; 0. bölüm bu oturumda çalışmadıysa önce onu çalıştırır)
VERI_KAYNAGI = "web"   #@param ["kütüphane", "web", "bilgisayar"]
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
case = vclinic.patient_ecg(clinic, "VC-0317")
print("2. durağın verileri hazır.")

#### Prompt 2.1: Bir D2 şeridi

**Amaç.** Bu promptla D2 derivasyonunun ilk 2,5 saniyesi çizilir ve böyle bir şeridin kaç sayıdan oluştuğu görülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: 67 yaşında bir kadın hastanın 10 saniyelik istirahat EKG'sinden D2 derivasyonunun ilk 2,5 saniyesini görmek ve bu kısa şeridin kaç örnekten oluştuğunu öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var. Her satır kaydın bir anıdır: "zaman" sütunu saniyeyi, "D2" sütunu D2 derivasyonundaki gerilimi milivolt olarak verir. Kayıt saniyede 500 kez örneklenmiştir.  
> Görev: Zamanı 2,5 saniyeden küçük olan satırları ayır ve say. Bu bölümü, yatay eksende saniye cinsinden zaman, dikey eksende milivolt cinsinden gerilim olacak biçimde çiz.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Bu 2,5 saniyedeki örnek sayısını yazdır ve "cevap" adıyla sakla. Grafiği de göster.  
> Test: Grafiğin zaman ekseni 0'da başlamalı ve 2,5 saniyede bitmeli.  

In [ ]:
#@title Prompt 2.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["case", "clinic"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
    case = vclinic.patient_ecg(clinic, "VC-0317")
veri = pd.DataFrame({"zaman": case.time, "D2": case.lead("II")})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Prompt 2.1 kontrolü
teach.check_step("Prompt 2.1", "cevap", globals())

In [ ]:
#@title Prompt 2.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["case", "clinic"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
    case = vclinic.patient_ecg(clinic, "VC-0317")
veri = pd.DataFrame({"zaman": case.time, "D2": case.lead("II")})
serit = veri[veri["zaman"] < 2.5]
cevap = len(serit)
fig, ax = plt.subplots(figsize=(8, 2.6))
ax.plot(serit["zaman"], serit["D2"], color="k", lw=0.9)
ax.set_xlim(0, 2.5)
ax.set_xlabel("Zaman (s)")
ax.set_ylabel("mV")
plt.show()
print("2,5 saniyedeki örnek sayısı:", cevap)
teach.check_step("Prompt 2.1", "cevap", globals())

**Sonuç ne gösteriyor?** Saniyede 500 örnekle 2,5 saniye, her derivasyon için 1.250 sayı demektir; 12 derivasyonlu 10 saniyelik kaydın tamamı ise 60.000 sayı tutar. Ekrandaki şerit, bu tablonun bir resmidir.

#### Prompt 2.2: R tepelerinden kalp hızı

**Amaç.** Bu promptla D2 derivasyonunun R tepeleri bulunur ve aralarındaki sürelerden kalp hızı hesaplanır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: 67 yaşında bir kadın hastanın 10 saniyelik istirahat EKG'sinden kalp hızını ölçmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var. Her satır kaydın bir anıdır: "zaman" sütunu saniyeyi, "D2" sütunu D2 derivasyonundaki gerilimi milivolt olarak verir. Kayıt saniyede 500 kez örneklenmiştir.  
> Görev: R tepelerini bul. En yüksek değerin yarısını aşan ve birbirinden en az 0,3 saniye uzakta duran tepeleri R tepesi kabul et. Ardışık R tepeleri arasındaki ortalama süreden kalp hızını dakikadaki atım sayısı olarak hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Kalp hızını bir ondalık basamakla yazdır ve "cevap" adıyla sakla. D2 derivasyonunu, R tepeleri işaretli olarak çiz.  
> Test: 10 saniyede yaklaşık 14 R tepesi bulunmalı; kalp hızı dakikada 40 ile 150 atım arasında olmalı.  

In [ ]:
#@title Prompt 2.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["case", "clinic"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
    case = vclinic.patient_ecg(clinic, "VC-0317")
veri = pd.DataFrame({"zaman": case.time, "D2": case.lead("II")})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Prompt 2.2 kontrolü
teach.check_step("Prompt 2.2", "cevap", globals())

In [ ]:
#@title Prompt 2.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["case", "clinic"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
    case = vclinic.patient_ecg(clinic, "VC-0317")
veri = pd.DataFrame({"zaman": case.time, "D2": case.lead("II")})
from scipy.signal import find_peaks
t = veri["zaman"].to_numpy()
x = veri["D2"].to_numpy()
r, _ = find_peaks(x, height=x.max() / 2, distance=int(0.3 * 500))
cevap = round(60 / np.diff(t[r]).mean(), 1)
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.plot(t, x, color="k", lw=0.7)
ax.plot(t[r], x[r], "o", color="#B4312A")
ax.set_xlabel("Zaman (s)")
ax.set_ylabel("mV")
plt.show()
print("R tepeleri:", len(r), " kalp hızı (atım/dk):", cevap)
teach.check_step("Prompt 2.2", "cevap", globals())

**Sonuç ne gösteriyor?** Vakanın kalp hızı, kayıttaki değere (dakikada 85 atım) yakındır. Promptta verilen kural basittir: T dalgaları yüksek bir hastada T dalgalarını da R tepesi sayabilir ve kalp hızını iki katına çıkarabilir. Ders sayfası bu durumu başka hastaların EKG'leriyle göstermektedir.

#### Prompt 2.3: İki formülle QTc

**Amaç.** Bu promptla VC-0254 numaralı hastanın QT aralığı, yüksek kalp hızına göre Fridericia ve Bazett formülleriyle düzeltilir ve iki sonuç karşılaştırılır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Kalp hızı yüksek olan 58 yaşında bir erkek hastanın QT aralığını iki formülle düzeltmek ve sonuçları karşılaştırmak istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında tek satırlık bir tablo var: "hasta" sütunu hastanın numarasını, "kalp hızı" sütunu dakikadaki atım sayısını, "QT" sütunu da QT aralığını milisaniye olarak verir.  
> Görev: RR aralığını saniye olarak, 60'ın kalp hızına bölümüyle hesapla. Ardından QT aralığını iki formülle düzelt: Fridericia formülü QT'yi RR'nin küp köküne, Bazett formülü ise RR'nin kareköküne böler.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Fridericia formülüyle bulunan QTc'yi milisaniye olarak, bir ondalık basamakla yazdır ve "cevap" adıyla sakla. Bazett formülünün sonucunu da yazdır.  
> Test: Kalp hızı dakikada 60 atım olduğunda iki formül de QT'nin kendisini vermeli.  

In [ ]:
#@title Prompt 2.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["case", "clinic"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
    case = vclinic.patient_ecg(clinic, "VC-0317")
veri = pd.DataFrame({"hasta": ["VC-0254"], "kalp hızı": [98], "QT": [343]})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Prompt 2.3 kontrolü
teach.check_step("Prompt 2.3", "cevap", globals())

In [ ]:
#@title Prompt 2.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["case", "clinic"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
    case = vclinic.patient_ecg(clinic, "VC-0317")
veri = pd.DataFrame({"hasta": ["VC-0254"], "kalp hızı": [98], "QT": [343]})
hiz = veri["kalp hızı"].iloc[0]
qt = veri["QT"].iloc[0]
rr = 60 / hiz
cevap = round(qt / rr ** (1 / 3), 1)
bazett = round(qt / rr ** 0.5, 1)
print("RR (s):", round(rr, 3), " QTc Fridericia:", cevap, " QTc Bazett:", bazett)
teach.check_step("Prompt 2.3", "cevap", globals())

**Sonuç ne gösteriyor?** Dakikada 98 atımda Bazett formülü, Fridericia formülünden yaklaşık 35 ms daha uzun bir QTc verir. Bazett formülü yüksek kalp hızlarında fazla düzeltme yapar; ilaç güvenliği çalışmalarının çoğu bu nedenle Fridericia düzeltmesini bildirmektedir.

**«Durak 2 · Prompt ile uygulama» tamamlandı.** Sırada hemen aşağıdaki «Durak 2 · Hazır uygulama» bölümü var. Durağı burada bırakacaksanız [ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#real-data-physionet-records) ve okumaya «PhysioNet ve MIT-BIH veri setlerine erişim» bölümünden devam edin.

### Durak 2 · Hazır uygulama

Bu bölüm haftanın vakasının EKG'sini veri olarak ele alır: Örnekler ve örnekleme frekansı, tek bir kalp vektöründen on iki derivasyon ve PR, QRS ve QT aralıklarının beş adımda ölçülmesi. Ölçüm 100 hastada ve sizin kaliper ölçümlerinizle sınanır.

Hücreleri yukarıdan aşağıya çalıştırın. *Python notu* hücreleri kodu açıklar, *Gözlemle ve düşün* hücreleri ise sonuçlar üzerine sorular sorar.

In [ ]:
#@title Hazır uygulamanın hazırlığı: Bu hücreyi bir kez çalıştırın (0. bölüm ve uygulamanın ihtiyaç duyduğu önceki kodlar bu oturumda çalışmadıysa onları çalıştırır)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    _catch_up(1, "0. bölüm çalıştı.")
print("2. durağın hazır uygulaması hazır.")

#### Veri olarak EKG: Örnekler, örnekleme frekansı ve birimler

*Ders: [Veri olarak EKG](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#the-ecg-as-data)*

**Formüllerden koda.** $f_s$ örnekleme frekansı `case.fs` içindedir. $n$ numaralı örneğin zamanı, yani $t_n = n / f_s$, `case.time` içinde tutulur. Bir aralıktaki örnek sayısı $N = \Delta t \cdot f_s$ ise `round(dt * case.fs)` ile hesaplanır. Milivolt cinsinden gerilimler `case.leads` tablosunun sütunlarıdır.

In [ ]:
clinic = vclinic.make_cohort(n=500, seed=2026)   # sanal klinik, 1. duraktaki gibi
case = vclinic.patient_ecg(clinic, "VC-0317")   # haftanın vakası
print(case)
print("derivasyon başına örnek sayısı:", len(case.leads))
print("zaman adımı (s):", 1 / case.fs)
case.leads.head()

In [ ]:
# EKG kâğıdında D2 derivasyonu: 10 saniye ve tek atım
fig, axes = plt.subplots(2, 1, figsize=(12, 6.5))
teach.plot_strip(axes[0], case.time, case.lead("II"), label="II")
one = (case.time >= 1.0) & (case.time <= 2.0)
teach.plot_strip(axes[1], case.time[one], case.lead("II")[one], label="II")
axes[1].plot(case.time[one][::10], case.lead("II")[one][::10], "o",
             color="#9B1B1B", ms=3, label="her 10. örnek")
axes[1].legend(loc="lower right")
plt.tight_layout()
plt.show()

**Python notu.** `case.time >= 1.0` ifadesi her zaman değerini `1.0` ile karşılaştırır ve `True` ile `False` değerlerinden oluşan bir dizi, yani bir *maske* döndürür. `&` ile birleştirilen iki maske, 1 ile 2 saniye arasındaki örnekleri işaretler ve `x[mask]` bu örnekleri seçer. `[::10]` ise her onuncu elemanı alır.

In [ ]:
# Üç örnekleme frekansında bir RR aralığındaki örnek sayısı
rr = 60 / 85
for fs in [500, 360, 1000]:
    print(f"fs = {fs} Hz'de {rr:.3f} s'lik RR aralığı {round(rr * fs)} örnek içerir")

**Keşfet.** Panel, vakanın bir kalp atımını bir analog-dijital dönüştürücünün (ADC) yapacağı gibi daha düşük bir frekansta yeniden örnekler ve daha az bitle saklar. Karşılaştırma için aynı atımın 2.000 Hz'lik hâli gösterilir. Kaydırıcıları hareket ettirin ve R tepesinin yüksekliğinin ve zamanının nasıl değiştiğine bakın. Yaklaşık 100 Hz'in altında R tepesi kırpılır ve zamanı kayar. Bit sayısı azaldıkça çizgi basamaklı bir görünüm alır.

In [ ]:
# Keşfet: örnekleme frekansı ve çözünürlük
fine = vclinic.simulate(case.params, duration_s=2.0, fs=2000, noise="none", seed=7)
t_ref, x_ref = fine.time, fine.lead("II")


def show_sampling(fs_hz=500, bits=12):
    step = int(round(2000 / fs_hz))
    t_s, x_s = t_ref[::step], x_ref[::step]
    q = 10.0 / 2 ** bits                       # 10 mV aralıkta adım büyüklüğü
    x_q = np.round(x_s / q) * q
    window = (t_ref > 0.25) & (t_ref < 0.75)
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.plot(t_ref[window], x_ref[window], color="#888888", lw=1,
            label="2000 Hz referans")
    w2 = (t_s > 0.25) & (t_s < 0.75)
    ax.step(t_s[w2], x_q[w2], where="mid", color="#9B1B1B", lw=1.2,
            label=f"{fs_hz} Hz, {bits} bit")
    ax.set_xlabel("Zaman (s)")
    ax.set_ylabel("mV")
    ax.legend(loc="upper right")
    plt.show()
    print(f"R tepesi: referans {x_ref.max():.3f} mV, "
          f"örneklenmiş {x_q.max():.3f} mV; "
          f"zaman kayması {abs(t_s[np.argmax(x_q)] - t_ref[np.argmax(x_ref)]) * 1000:.1f} ms; "
          f"çözünürlük {q * 1000:.2f} µV")


try:
    from ipywidgets import interact, IntSlider
    interact(show_sampling,
             fs_hz=IntSlider(value=500, min=50, max=1000, step=50, description="fs (Hz)"),
             bits=IntSlider(value=12, min=4, max=16, step=1, description="bits"))
except ImportError:
    show_sampling(500, 12)
    show_sampling(100, 6)

#### Tek kalp vektöründen on iki derivasyon

*Ders: [Vektör olarak kalp: On iki derivasyon, tek kaynak](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#the-heart-as-a-vector-twelve-leads-one-source)*

**Formüllerden koda.** $\mathbf{H} = (H_x, H_y, H_z)$ kalp vektörü, üç sayıdan oluşan bir NumPy dizisidir. On iki derivasyonun $\mathbf{L}$ derivasyon vektörleri `vclinic.LEAD_MATRIX` matrisinin satırlarıdır. Bütün derivasyonlar için $V_L = \mathbf{L} \cdot \mathbf{H}$ gerilimi tek seferde `vclinic.project(H)` ile hesaplanır; bu fonksiyon `LEAD_MATRIX @ H` işlemini yapar. Frontal eksen $\alpha = \operatorname{atan2}(H_y, H_x)$ ise `vclinic.frontal_axis(V_I, V_aVF)` ile bulunur.

In [ ]:
# Sanal kliniğin derivasyon vektörleri: x sola, y aşağı, z öne
lead_table = pd.DataFrame(vclinic.LEAD_MATRIX, index=vclinic.LEADS,
                          columns=["Lx", "Ly", "Lz"]).round(3)
lead_table

In [ ]:
# Dersteki çözümlü örnek
H = np.array([0.6, 0.8, -0.2])           # kalp vektörü (mV)
V = vclinic.project(H)
volts = pd.Series(V, index=vclinic.LEADS).round(3)
print(volts.to_string())
print("Einthoven: I + III =", round(volts["I"] + volts["III"], 3), "ve II =", volts["II"])
print("Goldberger: aVR + aVL + aVF =", round(volts["aVR"] + volts["aVL"] + volts["aVF"], 3))
print("Frontal eksen (derece):", round(vclinic.frontal_axis(V[0], V[5]), 1))

**Python notu.** `@` işareti bir matrisi bir vektörle çarpar: 12 × 3 boyutlu derivasyon matrisinin her satırı `H` içindeki üç sayıyla eleman eleman çarpılır ve çarpımlar toplanır. `pd.Series` burada derivasyon adlarıyla etiketlenmiş tek bir sütunu tutar.

In [ ]:
# Vakanın 12 derivasyonlu EKG'si
fig = teach.plot_12lead(case.time, case.leads, title="VC-0317")
plt.show()
L = case.leads
print("Einthoven yasasından en büyük sapma (mV):",
      float(np.abs(L["II"] - (L["I"] + L["III"])).max()))

**Keşfet.** Kalp vektörünü frontal düzlemde ve göğsün önüne ya da arkasına doğru döndürün ve on iki izdüşümü izleyin. Vektör bir derivasyonun yönüne yöneldiğinde o derivasyon en büyük pozitif sapmasını çizer; ters yöne yöneldiğinde ise negatif bir sapma çizer.

In [ ]:
# Keşfet: tek vektör, on iki izdüşüm
def show_projection(frontal=50, anterior=-15, size=1.3):
    h = size * vclinic.direction(frontal, anterior)
    v = vclinic.project(h)
    colors = ["#9B1B1B" if x >= 0 else "#1F5F8B" for x in v]
    fig, ax = plt.subplots(figsize=(9, 3.4))
    ax.bar(vclinic.LEADS, v, color=colors)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_ylim(-2.5, 2.5)
    ax.set_ylabel("İzdüşüm (mV)")
    ax.set_title(f"H = ({h[0]:.2f}, {h[1]:.2f}, {h[2]:.2f}) mV, eksen "
                 f"{vclinic.frontal_axis(v[0], v[5]):.0f}°")
    plt.show()


try:
    from ipywidgets import interact, IntSlider, FloatSlider
    interact(show_projection,
             frontal=IntSlider(value=50, min=-180, max=180, step=5, description="frontal (°)"),
             anterior=IntSlider(value=-15, min=-90, max=90, step=5, description="ön/arka (°)"),
             size=FloatSlider(value=1.3, min=0.2, max=2.0, step=0.1, description="büyüklük (mV)"))
except ImportError:
    show_projection(50, -15, 1.3)

**Gözlemle ve düşün.** Aynı vektör D2 derivasyonunda pozitif, V1 derivasyonunda ise negatif bir sapma veriyor. Farkı, vektör ile her derivasyon arasındaki açıyla açıklayın. Aynı büyüklükteki vektörler için göğüs derivasyonları neden ekstremite derivasyonlarından daha büyük sapmalar gösterir?

#### PR, QRS ve QT ölçümü

*Ders: [Aralıklar: PR, QRS ve QT](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#intervals-pr-qrs-and-qt)*

**Formüllerden koda.** $HR = 60 / RR$ kalp hızı `60 / rr_s` ile hesaplanır. Bazett düzeltmesi $QT_{cB} = QT / \sqrt{RR}$ için `qt / np.sqrt(rr_s)`, Fridericia düzeltmesi $QT_{cF} = QT / \sqrt[3]{RR}$ için ise `qt / np.cbrt(rr_s)` kullanılır; ikisinde de RR aralığı saniye cinsinden olmalıdır. Teğet yöntemiyle bulunan T dalgası sonu, $t_{end} = t_s + (b - x(t_s)) / x'(t_s)$, `t_end_tangent` fonksiyonunda `t_s + (base - x[t_s]) / slope * fs` satırıyla hesaplanır.

Bu bölümde ölçüm beş adımda kurulur ve adımlar sonunda tek bir fonksiyonda, `measure_ecg` içinde birleştirilir. Sonraki haftalarda kullanılmak üzere aynı adımlar `vclinic.measure` modülünde de yer almaktadır.

In [ ]:
# Adım 1. R tepeleri: bant geçiren filtre, türev, kare alma ve 150 ms'lik hareketli pencere
from scipy.signal import butter, filtfilt, find_peaks


def bandpass(x, fs, low, high):
    b, a = butter(2, [low / (fs / 2), high / (fs / 2)], btype="band")
    return filtfilt(b, a, x)


def lowpass(x, fs, cutoff=20.0):
    b, a = butter(2, cutoff / (fs / 2), btype="low")
    return filtfilt(b, a, x)


def detect_r_peaks(x, fs):
    y = bandpass(x, fs, 5, 15)                 # QRS frekanslarını korur
    d = np.gradient(y) * fs                    # eğim (mV/s)
    z = d ** 2                                 # yalnızca büyük eğimler öne çıkar
    w = int(round(0.150 * fs))                 # 150 ms'lik pencere
    m = np.convolve(z, np.ones(w) / w, mode="same")
    peaks, _ = find_peaks(m, height=0.3 * np.percentile(m, 99),
                          distance=int(0.25 * fs))
    yy = bandpass(x, fs, 1, 40)
    half = int(0.075 * fs)
    r = []
    for p in peaks:                            # 75 ms içindeki en büyük sapma
        lo, hi = max(0, p - half), min(len(x), p + half + 1)
        r.append(lo + int(np.argmax(np.abs(yy[lo:hi]))))
    return np.unique(r)


r_idx = detect_r_peaks(case.lead("II"), case.fs)
print("Bulunan R tepeleri:", len(r_idx))
print("zamanlar (s):", np.round(r_idx / case.fs, 3))
print("gerçek R zamanları (s):", np.round(case.truth["all_beats_s"], 3))

In [ ]:
# Adım 2. 12 derivasyonun eğiminden QRS başlangıcı ve sonu
def global_slope(leads, fs):
    arr = np.column_stack([bandpass(leads[:, j], fs, 1, 40)
                           for j in range(leads.shape[1])])
    g = np.abs(np.gradient(arr, axis=0)).sum(axis=1) * fs
    w = max(1, int(round(0.010 * fs)))
    return np.convolve(g, np.ones(w) / w, mode="same")


def qrs_bounds(leads, fs, r_idx, frac=0.08):
    g = global_slope(leads, fs)
    onsets, offsets = [], []
    for r in r_idx:
        lo, hi = max(0, r - int(0.12 * fs)), min(len(g) - 1, r + int(0.14 * fs))
        level = frac * g[lo:hi + 1].max()      # en büyük eğimin yüzde 8'i
        on = r
        while on > lo and g[on] > level:
            on -= 1
        off = r
        while off < hi and g[off] > level:
            off += 1
        onsets.append(on)
        offsets.append(off)
    return np.array(onsets), np.array(offsets)


on_idx, off_idx = qrs_bounds(case.leads.to_numpy(), case.fs, r_idx)
qrs_ms = (off_idx - on_idx) / case.fs * 1000
print("QRS süreleri (ms):", np.round(qrs_ms, 0))

**Python notu.** `while` döngüsü, koşulu doğru kaldığı sürece girintili satırlarını tekrarlar. Burada döngü R tepesinden başlar ve 12 derivasyonun eğimi eşiğin altına düşene kadar her adımda bir örnek geriye gider. `np.column_stack` dizileri bir matrisin sütunları olarak yan yana koyar.

In [ ]:
# Adım 3. Teğet yöntemiyle T dalgasının sonu (D2, 20 Hz alçak geçiren filtreden geçirilmiş)
def t_end_tangent(x, fs, qrs_off, next_r, base):
    lo = qrs_off + int(0.08 * fs)
    hi = qrs_off + int(0.7 * (next_r - qrs_off))
    seg = x[lo:hi] - base
    k = int(np.argmax(np.abs(seg)))
    peak = lo + k                              # T tepesi
    sign = 1.0 if seg[k] >= 0 else -1.0
    d = np.gradient(x) * fs
    stop = min(len(x) - 1, peak + int(0.20 * fs))
    t_s = peak + int(np.argmin(sign * d[peak:stop]))   # en dik nokta
    slope = d[t_s]
    if slope == 0:
        return np.nan
    return t_s + (base - x[t_s]) / slope * fs  # teğetin taban çizgisini kestiği yer


x20 = lowpass(case.lead("II"), case.fs, 20)
fs = case.fs
i = 2                                          # üçüncü atım
base = np.median(x20[on_idx[i] - int(0.03 * fs):on_idx[i]])
t_end = t_end_tangent(x20, fs, off_idx[i], r_idx[i + 1], base)
print(f"QT atımı: {(t_end - on_idx[i]) / fs * 1000:.1f} ms, gerçek: {case.truth['qt_ms']:.0f} ms")

In [ ]:
# Üçüncü atımda teğet çizimi
d20 = np.gradient(x20) * fs
seg = x20[off_idx[i] + int(0.08 * fs):off_idx[i] + int(0.7 * (r_idx[i + 1] - off_idx[i]))]
peak = off_idx[i] + int(0.08 * fs) + int(np.argmax(np.abs(seg - base)))
t_s = peak + int(np.argmin(d20[peak:peak + int(0.2 * fs)]))
tt = np.linspace(t_s / fs - 0.05, t_end / fs + 0.02, 50)
fig, ax = plt.subplots(figsize=(9, 4))
w = slice(on_idx[i] - int(0.1 * fs), int(t_end) + int(0.12 * fs))
teach.plot_strip(ax, case.time[w], x20[w], label="II", v_range=(-0.5, 1.5))
ax.plot(tt, x20[t_s] + d20[t_s] * (tt - t_s / fs), color="#9B1B1B", lw=1.5,
        label="en dik noktadaki teğet")
ax.axhline(base, color="#1F5F8B", lw=1, ls="--", label="taban çizgisi")
ax.axvline(on_idx[i] / fs, color="k", lw=1, ls=":")
ax.axvline(t_end / fs, color="#9B1B1B", lw=1, ls=":")
ax.legend(loc="upper right", fontsize=8)
plt.show()

In [ ]:
# Adım 4. P dalgasının başlangıcı
def p_onset(x, fs, qrs_on, earliest=0, frac=0.10):
    lo = max(0, int(earliest), qrs_on - int(0.30 * fs))
    hi = qrs_on - int(0.04 * fs)
    if hi - lo < int(0.08 * fs):               # bir P dalgası için çok kısa
        return np.nan
    seg = x[lo:hi]
    base = np.median(seg[:int(0.04 * fs)])     # P dalgasından önceki düz bölüm
    seg = seg - base
    k = int(np.argmax(seg))
    j = k
    while j > 0 and seg[j] > frac * seg[k]:    # P yüksekliğinin yüzde 10'una kadar geri
        j -= 1
    return lo + j


po = p_onset(x20, fs, on_idx[i], earliest=t_end_tangent(
    x20, fs, off_idx[i - 1], r_idx[i], base) + 0.02 * fs)
print(f"PR atımı: {(on_idx[i] - po) / fs * 1000:.0f} ms, gerçek: {case.truth['pr_ms']:.0f} ms")

In [ ]:
# Adım 5. Bütün adımlar tek fonksiyonda: EKG laboratuvarının ölçümü
def measure_ecg(leads, fs, lead="II", sinus=True):
    """12 derivasyonlu bir kaydın medyan kalp hızı, PR, QRS, QT ve QTc değerleri."""
    leads = np.asarray(leads, dtype=float)
    x = leads[:, ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3",
                  "V4", "V5", "V6"].index(lead)]
    r = detect_r_peaks(leads[:, 1], fs)
    on, off = qrs_bounds(leads, fs, r)
    x20 = lowpass(x, fs, 20)
    pr, qrs, qt = [], [], []
    t_prev = None
    for k in range(len(r) - 1):
        base = np.median(x20[max(0, on[k] - int(0.03 * fs)):on[k]])
        te = t_end_tangent(x20, fs, off[k], r[k + 1], base)
        qrs.append((off[k] - on[k]) / fs * 1000)
        qt.append((te - on[k]) / fs * 1000)
        if sinus and t_prev is not None and np.isfinite(t_prev):
            pr.append((on[k] - p_onset(x20, fs, on[k], t_prev + 0.02 * fs)) / fs * 1000)
        t_prev = te
    rr_s = np.mean(np.diff(r)) / fs
    out = {"hr_bpm": 60 / rr_s, "rr_s": rr_s,
           "pr_ms": np.nanmedian(pr) if np.isfinite(pr).any() else np.nan,
           "qrs_ms": np.median(qrs), "qt_ms": np.nanmedian(qt)}
    out["qtc_bazett_ms"] = out["qt_ms"] / np.sqrt(rr_s)
    out["qtc_fridericia_ms"] = out["qt_ms"] / np.cbrt(rr_s)
    return out


def compare(ecg):
    m = measure_ecg(ecg.leads.to_numpy(), ecg.fs, sinus=ecg.truth["rhythm"] == "sinus")
    keys = ["hr_bpm", "pr_ms", "qrs_ms", "qt_ms", "qtc_bazett_ms", "qtc_fridericia_ms"]
    return pd.DataFrame({"ölçülen": [m[k] for k in keys],
                         "gerçek": [ecg.truth[k] for k in keys]},
                        index=keys).round(1)


compare(case)

**Gözlemle ve düşün.** Haftanın vakasında Bazett formülü, kadınlar için belirlenen 460 ms eşiğinin üzerinde bir QTc vermektedir. Fridericia formülü ise bu eşiğin belirgin biçimde altında bir değer verir. Dakikada 85 atımda hangi düzeltmeyi rapora yazardınız ve neden? Ders sayfasının QT düzeltmesi bölümünde 2009 önerilerinin bu konudaki görüşü yer almaktadır.

In [ ]:
# Etkileşimli labın C bölümündeki beş hasta
LAB_PATIENTS = ["VC-0317", "VC-0079", "VC-0254", "VC-0188", "VC-0018"]
rows = []
for pid in LAB_PATIENTS:
    e = vclinic.patient_ecg(clinic, pid)
    m = measure_ecg(e.leads.to_numpy(), e.fs, sinus=e.truth["rhythm"] == "sinus")
    rows.append({"patient": pid, "conduction": e.truth["conduction"],
                 "HR": e.truth["hr_bpm"], "PR_true": e.truth["pr_ms"], "PR_alg": m["pr_ms"],
                 "QRS_true": e.truth["qrs_ms"], "QRS_alg": m["qrs_ms"],
                 "QT_true": e.truth["qt_ms"], "QT_alg": m["qt_ms"]})
lab_table = pd.DataFrame(rows).set_index("patient").round(1)
lab_table

Listedeki son hasta olan VC-0018'de sol dal bloğu (left bundle branch block, LBBB) vardır. QRS kompleksi geniştir; D2'deki T dalgası ise küçüktür ve QRS'nin ana yönüne zıttır. Bu yüzden D2'de teğet yöntemi bu hasta için güvenilir değildir; aşağıdaki *İleri düzey* hücresi bunun nedenini göstermektedir.

In [ ]:
# Klinikten 100 hasta üzerinde doğrulama (sinüs ritmi, normal iletim)
sample = clinic[(clinic["ecg_rhythm"] == "sinus")
                & (clinic["ecg_conduction"] == "normal")].head(100)
errors, qt_pairs = [], []
for pid in sample["patient_id"]:
    e = vclinic.patient_ecg(clinic, pid)
    m = measure_ecg(e.leads.to_numpy(), e.fs)
    errors.append({k: m[k] - e.truth[k] for k in ["pr_ms", "qrs_ms", "qt_ms"]})
    qt_pairs.append((m["qt_ms"], e.truth["qt_ms"]))
errors = pd.DataFrame(errors)
qt_pairs = np.array(qt_pairs)
summary = pd.DataFrame({"sapma (ms)": errors.mean(),
                        "SS (ms)": errors.std(),
                        "error| (ms)|en büyük |hata| (ms)": errors.abs().max()}).round(1)
summary

**Kaliper ölçümleriniz, algoritma ve gerçek değer.** Etkileşimli labın *Düşün ve dışa aktar* sekmesi, C bölümündeki kaliper ölçümlerinizi `my_caliper_measurements.csv` dosyası olarak dışa aktarır. Dosyayı bu Jupyter Notebook'un klasörüne kaydedin ve sonraki hücreyi çalıştırın. Dosya indirilemiyorsa *Kaliper ölçümlerini kopyala* düğmesi aynı metni kopyalar: Metni sonraki hücrede `PASTED_CALIPERS` değişkeninin üçlü tırnakları arasına yapıştırın. Kendi ölçümleriniz yoksa hücre, elle yapılmış ölçümleri taklit eden bir örnek dosya kullanır.

In [ ]:
# QT aralığının Bland-Altman analizi
import io
import os
from vclinic.datasets import data_path

# kopyalanan kaliper ölçümlerini üçlü tırnakların arasına yapıştırın
PASTED_CALIPERS = """
"""

own = "my_caliper_measurements.csv"
if PASTED_CALIPERS.strip():
    source = "PASTED_CALIPERS"
    calipers = pd.read_csv(io.StringIO(PASTED_CALIPERS.strip()))
elif os.path.exists(own):
    source = own
    calipers = pd.read_csv(own)
else:
    source = data_path("examples", "example_caliper_measurements.csv")
    calipers = pd.read_csv(source)
print("Ölçümlerin kaynağı:", source)
qt_manual = calipers[calipers["measure"] == "QT"].set_index("patient_id")["value_ms"]
pts = [p for p in qt_manual.index if p != "VC-0018"]
truth_qt = [vclinic.patient_ecg(clinic, p).truth["qt_ms"] for p in pts]
alg_qt = [lab_table.loc[p, "QT_alg"] for p in pts]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
teach.plot_bland_altman(axes[0], qt_manual[pts].to_numpy(), truth_qt)
axes[0].set_title("Kaliper eksi gerçek")
teach.plot_bland_altman(axes[1], qt_pairs[:, 0], qt_pairs[:, 1])
axes[1].set_title("Algoritma eksi gerçek, 100 hasta")
plt.tight_layout()
plt.show()

**Python notu.** `elif`, *değilse eğer* anlamına gelir: Python koşulları yukarıdan aşağıya sınar ve yalnızca koşulu doğru olan ilk bloğu çalıştırır. `io.StringIO` bir metnin dosya gibi okunmasını sağlar; böylece `pd.read_csv`, yapıştırılan metni diskteki bir dosyaymış gibi okuyabilir.

**Alıştırma 2.3.** VC-0188 numaralı hastanın PR aralığını `measure_ecg` ile ölçün ve en yakın milisaniyeye yuvarlayın. Sonuç, birinci derece atriyoventriküler blok tanımını karşılıyor mu?

In [ ]:
# Bu alıştırmanın promptu için "veri" tablosu (etkileşimli labın F bölümü)
ekg_g = vclinic.patient_ecg(clinic, "VC-0188")
olcum_g = measure_ecg(ekg_g.leads.to_numpy(), ekg_g.fs)
veri = pd.DataFrame({"kalp hızı": [round(olcum_g["hr_bpm"], 1)], "PR": [round(olcum_g["pr_ms"], 1)],
                     "QRS": [round(olcum_g["qrs_ms"], 1)], "QT": [round(olcum_g["qt_ms"], 1)]})
cevap = None   # cevabınız
teach.check("Alıştırma 2.3", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
e = vclinic.patient_ecg(clinic, "VC-0188")
cevap = round(measure_ecg(e.leads.to_numpy(), e.fs)["pr_ms"])
teach.check("Alıştırma 2.3", cevap)
print("Birinci derece blok (PR > 200 ms):", cevap > 200)

**İleri düzey (isteğe bağlı).** 2009 önerilerine göre QT, en uzun QT'yi gösteren derivasyonda ölçülmelidir; bu derivasyon çoğu zaman V2 ya da V3'tür. `measure_ecg` fonksiyonunu, QT'yi D2, V2, V3 ve V5 derivasyonlarında ölçecek ve en uzun değeri verecek biçimde genişletin. Fonksiyonu VC-0018 üzerinde deneyin ve her derivasyonun T dalgasına bakın: T dalgası QRS'ye zıt olduğunda teğet yöntemi neden zorlanır?

In [ ]:
# İleri düzey (isteğe bağlı): dört derivasyonda QT
e = vclinic.patient_ecg(clinic, "VC-0018")
for lead in ["II", "V2", "V3", "V5"]:
    print(lead, round(measure_ecg(e.leads.to_numpy(), e.fs, lead=lead)["qt_ms"], 1))
print("gerçek QT:", e.truth["qt_ms"])

**Promptların sonuçları.** Aşağıdaki hücre 2. durağın promptlarının sonuçlarını bu uygulamanın koduyla hesaplar ve promptların kontrol hücreleri gibi denetler.

In [ ]:
# Bu durağın promptlarının sonuçları, Python ile
samples = int(np.sum(case.time < 2.5))
print("D2'nin 2,5 saniyedeki örnek sayısı:", samples)
teach.check("Prompt 2.1", samples)
hr = round(60 / np.diff(r_idx / case.fs).mean(), 1)       # yukarıdaki 1. adımın R tepelerinden
print("Kalp hızı (atım/dk):", hr)
teach.check("Prompt 2.2", hr)
qtc_254 = round(343 / (60 / 98) ** (1 / 3), 1)             # VC-0254: Dakikada 98 atımda QT 343 ms
print("VC-0254'ün Fridericia QTc değeri (ms):", qtc_254)
teach.check("Prompt 2.3", qtc_254)

**2. durak tamamlandı.** [Ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#real-data-physionet-records) ve okumaya «PhysioNet ve MIT-BIH veri setlerine erişim» bölümünden devam edin. Bir sonraki uygulama durağında ders sayfası sizi yine bu Jupyter Notebook'a getirir.

## Durak 3: Gerçek kayıtlar ve özensiz bir prompt

*Ders sayfasında: [3. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#stop-3)*

Gerçek kayıtlar kendi gürültüsü, işaretlemeleri ve birimleriyle gelir. Bu durakta MIT-BIH Aritmi Veri Tabanı'ndaki 100 numaralı kayıt, kardiyologların verdiği atım etiketleriyle birlikte okunur ve normal atımlardan kalp hızı değişkenliği hesaplanır. Ardından PTB veri tabanındaki gerçek bir 12 derivasyonlu EKG çizilir. Son olarak, bir YZ asistanının kısa ve özensiz bir prompttan yazdığı QTc hesabı prompt kartıyla yeniden istenir ve sonucu denetlenir.

**Durağın sonunda:** 100 numaralı kayıttaki atriyal erken atımların sayısını (33), normal atımlar arasındaki aralıkların SDNN değerini (36,0 ms), PTB kaydının kalp hızını (81,8 atım/dk) ve düzeltilmiş QTc değerini (466,5 ms) elde edeceksiniz.

Durak üç bölümden oluşur: **Python temelleri**, **Prompt ile uygulama** ve **Hazır uygulama**. Bölümleri bu sırayla izleyin. Python biliyorsanız promptlarla, yalnızca sonucu görmek istiyorsanız hazır uygulamayla başlayabilirsiniz. Her bölümün ilk hücresi, bölümün ihtiyaç duyduğu her şeyi hazırlar.

### Durak 3 · Python temelleri

*Ders sayfasında: [PhysioNet ve MIT-BIH veri setlerine erişim](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#real-data-physionet-records)*

Gerçek bir kayıt, sinyalin yanında etiketler de taşır: Her atımın nerede olduğu ve ne tür bir atım olduğu. Bu bölümün küçük örnekleri, bu tür bilgilerle çalışmak için gereken Python'u öğretir. Bölümde etiketlerin sayılması, atımlar arasındaki aralıklar, kalp hızı değişkenliğinin istatistikleri, bir kaydın öznitelikleri, hataların yakalanması, varsayılan değerler ve bir şekil ızgarası ele alınır.

Hücreleri sırayla çalıştırın. Bir örneğin altındaki *Python notu* yeni yapıyı açıklar. Alıştırmalar hemen *doğru* ya da *henüz değil* yazar; örnek çözümler katlanmış hücrelerdedir.

In [ ]:
#@title Python temellerinin hazırlığı: Bu hücreyi bir kez çalıştırın (0. bölüm bu oturumda çalışmadıysa onu çalıştırır)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
print("3. durağın Python temelleri hazır.")

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
labels = ["N", "N", "A", "N", "N", "V", "N", "A", "N", "N"]    # N normal, A atriyal erken, V ventriküler erken atım
print("atım sayısı:", len(labels))
print("atriyal erken atım sayısı:", labels.count("A"))
print(pd.Series(labels).value_counts())

**Python notu.** `labels.count("A")` bir değerin listede kaç kez geçtiğini sayar. `pd.Series(labels)` listeyi bir pandas sütununa çevirir; `value_counts()` ise bütün değerleri tek seferde, en sık olandan en seyrek olana doğru sayar.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
meaning = {"N": "normal", "A": "atriyal erken", "V": "ventriküler erken"}
for symbol in sorted(set(labels)):
    print(symbol, "=", meaning[symbol], ":", labels.count(symbol))

**Python notu.** `set(labels)` her değeri bir kez tutar, `sorted` ise değerleri sıraya koyar. `meaning` sözlüğü, bir veri tabanının etiket rehberi gibi her sembolü sözcüklere çevirir.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
fs_rec = 360                                            # kaydın örnekleme frekansı (Hz)
r_samples = np.array([77, 370, 662, 946, 1231])         # beş R tepesinin örnek numaraları
rr_ms = np.diff(r_samples) / fs_rec * 1000              # komşu R tepeleri arasındaki aralıklar (ms)
print(rr_ms.round(1))
print("ortalama RR aralığı (ms):", round(rr_ms.mean(), 1))

**Python notu.** `np.diff` her elemanı bir sonrakinden çıkarır; bu yüzden beş R tepesinden dört aralık çıkar. Bir örnek sayısı örnekleme frekansına bölündüğünde saniyeyi, ardından 1000 ile çarpıldığında milisaniyeyi verir.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
rr = np.array([812, 805, 798, 820, 815, 790])           # altı RR aralığı (ms)
print("ortalama (ms):", round(rr.mean(), 1))
print("standart sapma (ms):", round(rr.std(ddof=1), 1))
diffs = np.diff(rr)
print("ardışık farklar (ms):", diffs)
print("farkların karelerinin ortalamasının karekökü (ms):", round(np.sqrt(np.mean(diffs ** 2)), 1))

**Python notu.** `rr.std(ddof=1)` paydasında n − 1 bulunan örneklem standart sapmasıdır; SDNN formülünde de böyledir. `diffs ** 2` her elemanın karesini alır, `np.mean` karelerin ortalamasını, `np.sqrt` ise karekökünü hesaplar: Üçü birlikte ders sayfasındaki RMSSD'yi verir.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
from types import SimpleNamespace

record_demo = SimpleNamespace(record_name="100", fs=360, sig_name=["MLII", "V5"], sig_len=650000)
print(record_demo.record_name, record_demo.sig_name)
print("süre (dk):", round(record_demo.sig_len / record_demo.fs / 60, 1))

**Python notu.** Bazı kütüphaneler verilerini adlandırılmış özniteliklerde tutan bir *nesne* döndürür; bu öznitelikler nokta ile okunur: `record_demo.fs` örnekleme frekansıdır. wfdb kayıtlarını bu biçimde, `fs`, `sig_name`, `sig_len` ve `p_signal` öznitelikleriyle döndürür. `SimpleNamespace` böyle bir nesneyi elle kurar; burada alıştırma için kullanılmıştır.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
text = "12,5"                                           # ondalık virgülle yazılmış bir sayı
try:
    value = float(text)
except ValueError:
    value = float(text.replace(",", "."))               # Python ondalık nokta bekler
print(value)

**Python notu.** `try` bir bloğu çalıştırır; `except` ise adı verilen türdeki bir hatayı, hücreyi durdurmadan yakalar. `float("12,5")` hata verir, çünkü Python ondalık sayıları nokta ile yazar; `except` bloğu metni düzeltir. İnternetten bir kayıt okumak da aynı kalıbı kullanır: Bağlantı yoksa `except` bilgisayardaki bir kopyaya yönelebilir.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
def samples_to_ms(n_samples, fs=360):
    return n_samples / fs * 1000


print(samples_to_ms(180))               # varsayılan fs = 360 kullanılır
print(samples_to_ms(180, fs=500))       # 500 Hz'lik bir kayıt

**Python notu.** `fs=360` bir *varsayılan değerdir*: Çağrıda `fs` verilmezse fonksiyon bu değeri kullanır. Varsayılan değer kolaylık sağlar, ama kod güvenlik kontrol listesindeki sessiz hatanın da kaynağıdır: 180 örnek, 360 Hz'de 500 ms, 500 Hz'de ise 360 ms eder.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
t3 = np.arange(0, 1, 1 / 250)
fig, axes = plt.subplots(2, 3, figsize=(8, 3.6), sharex=True, sharey=True)
for k, ax in enumerate(axes.flat):
    ax.plot(t3, np.sin(2 * np.pi * (k + 1) * t3), lw=0.8)
    ax.set_title(f"dalga {k + 1}", fontsize=9)
fig.supxlabel("zaman (s)")
plt.tight_layout()
plt.show()

**Python notu.** `plt.subplots(2, 3)` iki satır ve üç sütundan oluşan bir eksen ızgarası kurar. `axes.flat` eksenleri tek tek dolaşır, `enumerate` ise her birinin sırasını verir. 12 derivasyonlu EKG'nin standart çıktısı da üç satır ve dört sütunlu böyle bir ızgaradır.

**Alıştırma 3.1.** 1000, 960, 1010, 990 ve 1030 ms'lik RR aralıklarının RMSSD değerini hesaplayın ve sonucu iki ondalık basamağa yuvarlayın.

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma 3.1", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
rr_ex = np.array([1000, 960, 1010, 990, 1030])
cevap = round(float(np.sqrt(np.mean(np.diff(rr_ex) ** 2))), 2)
teach.check("Alıştırma 3.1", cevap)

**Alıştırma 3.2.** Saniyede 250 kez örneklenmiş bir EKG'de iki komşu R tepesi 410. ve 612. örnektedir. RR aralığını milisaniye olarak hesaplayın ve bir ondalık basamağa yuvarlayın.

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma 3.2", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round((612 - 410) / 250 * 1000, 1)
teach.check("Alıştırma 3.2", cevap)

**«Durak 3 · Python temelleri» tamamlandı.** Sırada hemen aşağıdaki «Durak 3 · Prompt ile uygulama» bölümü var. Durağı burada bırakacaksanız [ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#the-paper-of-the-week) ve okumaya «Haftanın makalesi» bölümünden devam edin.

### Durak 3 · Prompt ile uygulama

Bu bölümün dört promptu PhysioNet'in gerçek kayıtlarıyla çalışır; sonuncusu özensiz bir promptu prompt kartına dönüştürür. Onarım promptu ve hata mesajı için prompt, [prompt kütüphanesinde](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/PROMPTLAR.md) ve 1. durağın promptlarında yer alır.

Her promptta dört adım vardır. Promptu kopyalayıp YZ asistanınıza gönderin. Altındaki veri hücresini çalıştırın; bu hücre, promptun tarif ettiği "veri" tablosunu hazırlar. Asistanın kodunu boş hücreye yapıştırıp çalıştırın. Ardından kontrol hücresini çalıştırın; hücre *doğru* ya da *henüz değil* yazar. Asistanın kodu kontrolden geçmezse *Hazır kod* hücresini çalıştırıp devam edin.

In [ ]:
#@title Promptların hazırlığı: Bu hücreyi bir kez çalıştırın (verileri yükler; 0. bölüm bu oturumda çalışmadıysa önce onu çalıştırır)
VERI_KAYNAGI = "web"   #@param ["kütüphane", "web", "bilgisayar"]
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
from vclinic.datasets import read_physionet, read_physionet_annotation
mit, fs_mit, notes = read_physionet("100", "mitdb", source=VERI_KAYNAGI)
ann_samples, ann_symbols = read_physionet_annotation("100", "mitdb", source=VERI_KAYNAGI)
ptb, fs_ptb, header = read_physionet("s0010_re", "ptbdb/patient001", source=VERI_KAYNAGI)
print("3. durağın verileri hazır.")

#### Prompt 3.1: 100 numaralı kaydın atımları

**Amaç.** Bu promptla 30 dakikalık bir kaydın atımları, kardiyologların onlara verdiği etiketlere göre sayılır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: MIT-BIH Aritmi Veri Tabanı'ndaki 100 numaralı kayıt, 69 yaşında bir erkeğin 30 dakikalık ambulatuvar EKG kaydıdır. Bu kayıttaki atımları, kardiyologların onlara verdiği etiketlere göre saymak istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var. Her satır, kardiyologların kayda koyduğu bir işarettir: "zaman" sütunu işaretin yerini saniye olarak, "etiket" sütunu da türünü verir. "N" normal atımı, "A" atriyal erken atımı, "V" ventriküler erken atımı gösterir; "+" ise ritim değişikliğini belirtir ve bir atım değildir.  
> Görev: Her etiketin kaç kez geçtiğini say ve sayıları yazdır.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Atriyal erken atımların sayısını yazdır ve "cevap" adıyla sakla.  
> Test: N, A ve V etiketlerinin toplamı 2.273 atım olmalı.  

In [ ]:
#@title Prompt 3.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["ann_samples", "ann_symbols", "fs_mit", "fs_ptb", "header", "mit", "notes", "ptb", "read_physionet", "read_physionet_annotation"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    from vclinic.datasets import read_physionet, read_physionet_annotation
    mit, fs_mit, notes = read_physionet("100", "mitdb", source=VERI_KAYNAGI)
    ann_samples, ann_symbols = read_physionet_annotation("100", "mitdb", source=VERI_KAYNAGI)
    ptb, fs_ptb, header = read_physionet("s0010_re", "ptbdb/patient001", source=VERI_KAYNAGI)
veri = pd.DataFrame({"zaman": np.asarray(ann_samples) / fs_mit, "etiket": ann_symbols})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Prompt 3.1 kontrolü
teach.check_step("Prompt 3.1", "cevap", globals())

In [ ]:
#@title Prompt 3.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["ann_samples", "ann_symbols", "fs_mit", "fs_ptb", "header", "mit", "notes", "ptb", "read_physionet", "read_physionet_annotation"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    from vclinic.datasets import read_physionet, read_physionet_annotation
    mit, fs_mit, notes = read_physionet("100", "mitdb", source=VERI_KAYNAGI)
    ann_samples, ann_symbols = read_physionet_annotation("100", "mitdb", source=VERI_KAYNAGI)
    ptb, fs_ptb, header = read_physionet("s0010_re", "ptbdb/patient001", source=VERI_KAYNAGI)
veri = pd.DataFrame({"zaman": np.asarray(ann_samples) / fs_mit, "etiket": ann_symbols})
sayilar = veri["etiket"].value_counts()
print(sayilar)
cevap = int(sayilar.get("A", 0))
print("Atriyal erken atımlar:", cevap)
teach.check_step("Prompt 3.1", "cevap", globals())

**Sonuç ne gösteriyor?** 100 numaralı kayıt, 69 yaşında bir erkeğe ait sakin bir kayıttır: Atımların çoğu normaldir; kayıtta birkaç düzine atriyal erken atım ve tek bir ventriküler erken atım vardır. Bu etiketler kaydı, bir algoritmanın sınanabileceği bir referansa dönüştürür.

#### Prompt 3.2: Kalp hızı değişkenliği: SDNN

**Amaç.** Bu promptla normal atımlar arasındaki sürelerin 30 dakika boyunca ne kadar değiştiği ölçülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: MIT-BIH Aritmi Veri Tabanı'ndaki 100 numaralı kayıtta, yani 69 yaşında bir erkeğin 30 dakikalık ambulatuvar EKG'sinde, kalp hızı değişkenliğini SDNN ile ölçmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var. Her satır, kardiyologların kayda koyduğu bir işarettir: "zaman" sütunu işaretin yerini saniye olarak, "etiket" sütunu da türünü verir. "N" normal atımı, "A" atriyal erken atımı, "V" ventriküler erken atımı gösterir; "+" ise ritim değişikliğini belirtir ve bir atım değildir.  
> Görev: Yalnızca atımları, yani "+" dışındaki etiketleri tut. Ardışık atımlar arasındaki RR aralıklarını milisaniye olarak hesapla. Bunlardan yalnızca iki ucu da normal atım ("N") olan aralıkları, yani NN aralıklarını kullan. SDNN, NN aralıklarının standart sapmasıdır; bu sapmayı bir örneklem için olduğu gibi, paydada aralık sayısının bir eksiğiyle hesapla. Ortalama NN aralığını da yazdır.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: SDNN değerini milisaniye olarak, bir ondalık basamakla yazdır ve "cevap" adıyla sakla.  
> Test: Ortalama NN aralığı 700 ile 900 ms arasında olmalı.  

In [ ]:
#@title Prompt 3.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["ann_samples", "ann_symbols", "fs_mit", "fs_ptb", "header", "mit", "notes", "ptb", "read_physionet", "read_physionet_annotation"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    from vclinic.datasets import read_physionet, read_physionet_annotation
    mit, fs_mit, notes = read_physionet("100", "mitdb", source=VERI_KAYNAGI)
    ann_samples, ann_symbols = read_physionet_annotation("100", "mitdb", source=VERI_KAYNAGI)
    ptb, fs_ptb, header = read_physionet("s0010_re", "ptbdb/patient001", source=VERI_KAYNAGI)
veri = pd.DataFrame({"zaman": np.asarray(ann_samples) / fs_mit, "etiket": ann_symbols})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Prompt 3.2 kontrolü
teach.check_step("Prompt 3.2", "cevap", globals())

In [ ]:
#@title Prompt 3.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["ann_samples", "ann_symbols", "fs_mit", "fs_ptb", "header", "mit", "notes", "ptb", "read_physionet", "read_physionet_annotation"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    from vclinic.datasets import read_physionet, read_physionet_annotation
    mit, fs_mit, notes = read_physionet("100", "mitdb", source=VERI_KAYNAGI)
    ann_samples, ann_symbols = read_physionet_annotation("100", "mitdb", source=VERI_KAYNAGI)
    ptb, fs_ptb, header = read_physionet("s0010_re", "ptbdb/patient001", source=VERI_KAYNAGI)
veri = pd.DataFrame({"zaman": np.asarray(ann_samples) / fs_mit, "etiket": ann_symbols})
atimlar = veri[veri["etiket"] != "+"]
etiketler = atimlar["etiket"].to_numpy()
rr_ms = np.diff(atimlar["zaman"].to_numpy()) * 1000
nn_ms = rr_ms[(etiketler[:-1] == "N") & (etiketler[1:] == "N")]
cevap = round(np.std(nn_ms, ddof=1), 1)
print("NN aralıkları:", len(nn_ms), " ortalama (ms):", round(nn_ms.mean(), 1), " SDNN (ms):", cevap)
teach.check_step("Prompt 3.2", "cevap", globals())

**Sonuç ne gösteriyor?** SDNN kaydın süresine bağlıdır; bu yüzden 30 dakikalık bir değer, 5 dakikalık ya da 24 saatlik bir değerle karşılaştırılamaz. 1996 standartları 5 dakikalık ve 24 saatlik kayıtlar için ayrı referans değerler vermektedir.

#### Prompt 3.3: Gerçek bir 12 derivasyonlu EKG

**Amaç.** Bu promptla akut miyokard enfarktüsü geçiren bir hastanın gerçek 12 derivasyonlu EKG'si, EKG kâğıdındaki düzende çizilir ve kalp hızı ölçülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Akut infero-lateral miyokard enfarktüsüyle yatırılmış 81 yaşında bir kadın hastanın gerçek 12 derivasyonlu EKG'sini kâğıttaki düzende görmek ve kalp hızını ölçmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var. Her satır kaydın bir anıdır: "zaman" sütunu saniyeyi; "D1", "D2", "D3", "aVR", "aVL", "aVF" ve "V1"-"V6" sütunları da on iki standart derivasyondaki gerilimi milivolt olarak verir. Kayıt 38,4 saniye sürer ve saniyede 1.000 kez örneklenmiştir.  
> Görev: On iki derivasyonun ilk 3 saniyesini EKG kâğıdındaki gibi üç satır ve dört sütun hâlinde çiz: Birinci sütunda D1, D2 ve D3; ikincide aVR, aVL ve aVF; üçüncüde V1-V3; dördüncüde V4-V6 yer alsın ve her panelin üstünde derivasyonun adı yazsın. Ardından D1 derivasyonunda R tepelerini bul. En yüksek değerin yarısını aşan ve birbirinden en az 0,3 saniye uzakta duran tepeleri R tepesi kabul et. Ardışık R tepeleri arasındaki ortalama süreden kalp hızını dakikadaki atım sayısı olarak hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Kalp hızını bir ondalık basamakla yazdır ve "cevap" adıyla sakla. Şekli de göster.  
> Test: Kalp hızı dakikada 40 ile 150 atım arasında olmalı ve R tepelerinin sayısı bu hızla uyumlu olmalı: Dakikada 80 atımla 38,4 saniyede yaklaşık 51 atım bulunur.  

In [ ]:
#@title Prompt 3.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["ann_samples", "ann_symbols", "fs_mit", "fs_ptb", "header", "mit", "notes", "ptb", "read_physionet", "read_physionet_annotation"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    from vclinic.datasets import read_physionet, read_physionet_annotation
    mit, fs_mit, notes = read_physionet("100", "mitdb", source=VERI_KAYNAGI)
    ann_samples, ann_symbols = read_physionet_annotation("100", "mitdb", source=VERI_KAYNAGI)
    ptb, fs_ptb, header = read_physionet("s0010_re", "ptbdb/patient001", source=VERI_KAYNAGI)
derivasyonlar = {"i": "D1", "ii": "D2", "iii": "D3", "avr": "aVR", "avl": "aVL", "avf": "aVF",
               "v1": "V1", "v2": "V2", "v3": "V3", "v4": "V4", "v5": "V5", "v6": "V6"}
veri = ptb[list(derivasyonlar)].rename(columns=derivasyonlar)
veri.insert(0, "zaman", np.arange(len(ptb)) / fs_ptb)
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Prompt 3.3 kontrolü
teach.check_step("Prompt 3.3", "cevap", globals())

In [ ]:
#@title Prompt 3.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["ann_samples", "ann_symbols", "fs_mit", "fs_ptb", "header", "mit", "notes", "ptb", "read_physionet", "read_physionet_annotation"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    from vclinic.datasets import read_physionet, read_physionet_annotation
    mit, fs_mit, notes = read_physionet("100", "mitdb", source=VERI_KAYNAGI)
    ann_samples, ann_symbols = read_physionet_annotation("100", "mitdb", source=VERI_KAYNAGI)
    ptb, fs_ptb, header = read_physionet("s0010_re", "ptbdb/patient001", source=VERI_KAYNAGI)
derivasyonlar = {"i": "D1", "ii": "D2", "iii": "D3", "avr": "aVR", "avl": "aVL", "avf": "aVF",
               "v1": "V1", "v2": "V2", "v3": "V3", "v4": "V4", "v5": "V5", "v6": "V6"}
veri = ptb[list(derivasyonlar)].rename(columns=derivasyonlar)
veri.insert(0, "zaman", np.arange(len(ptb)) / fs_ptb)
from scipy.signal import find_peaks
t = veri["zaman"].to_numpy()
duzen = ["D1", "D2", "D3", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
ilk3 = t < 3
fig, axes = plt.subplots(3, 4, figsize=(12, 6), sharex=True)
for ax, ad in zip(axes.T.ravel(), duzen):
    ax.plot(t[ilk3], veri[ad].to_numpy()[ilk3], color="k", lw=0.7)
    ax.set_title(ad, fontsize=9)
plt.tight_layout()
plt.show()
x = veri["D1"].to_numpy()
r, _ = find_peaks(x, height=x.max() / 2, distance=int(0.3 * 1000))
cevap = round(60 / np.diff(t[r]).mean(), 1)
print("R tepeleri:", len(r), " kalp hızı (atım/dk):", cevap)
teach.check_step("Prompt 3.3", "cevap", globals())

**Sonuç ne gösteriyor?** Şekilde inferior derivasyonların (D2, D3 ve aVF) ve lateral derivasyonların ST segmentlerine bakmak gerekir. Algoritma aralıkları ölçer; enfarktüs örüntüsüne karar vermek ise hekimin işidir. R tepeleri D1 derivasyonunda sayılmıştır; çünkü aynı basit kural bu kaydın D2 derivasyonunda atımların üçte birinden fazlasını kaçırır. Bir kural, uygulandığı derivasyonda denetlenmelidir.

#### Prompt 3.4: Özensiz bir prompttan prompt kartına

**Amaç.** Bir YZ asistanı, "örnek sayısı olarak verilen qt ve rr'den qtc veren bir python fonksiyonu yaz" biçimindeki kısa prompta karşılık 17 ms gibi olanaksız bir QTc veren bir kod yazmıştır. Bu promptun veri hücresi bu sonucu da göstermektedir. Bu promptla aynı hesap, prompt kartının yedi alanıyla yeniden istenir ve sonuç denetlenir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Saniyede 500 kez örneklenmiş bir EKG'de QT ve RR aralıkları örnek sayısı olarak ölçüldü. Bazett yöntemiyle düzeltilmiş QT aralığını milisaniye olarak bulmak istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında tek satırlık bir tablo var: "QT" sütunu QT aralığını, "RR" sütunu RR aralığını örnek sayısı olarak verir. Kayıt saniyede 500 kez örneklenmiştir.  
> Görev: QT aralığını milisaniyeye, RR aralığını saniyeye çevir. QT'yi Bazett yöntemiyle düzelt: Milisaniye cinsinden QT'yi, saniye cinsinden RR aralığının kareköküne böl.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Saniyedeki örnek sayısını kendin varsayma; burada verilen 500 değerini kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Düzeltilmiş QT'yi milisaniye olarak, bir ondalık basamakla yazdır ve "cevap" adıyla sakla.  
> Test: Aynı hesap, QT 200 örnek ve RR 500 örnek olduğunda tam 400 ms vermeli; çünkü bu durumda RR aralığı tam 1 saniyedir.  

In [ ]:
#@title Prompt 3.4 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["ann_samples", "ann_symbols", "fs_mit", "fs_ptb", "header", "mit", "notes", "ptb", "read_physionet", "read_physionet_annotation"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    from vclinic.datasets import read_physionet, read_physionet_annotation
    mit, fs_mit, notes = read_physionet("100", "mitdb", source=VERI_KAYNAGI)
    ann_samples, ann_symbols = read_physionet_annotation("100", "mitdb", source=VERI_KAYNAGI)
    ptb, fs_ptb, header = read_physionet("s0010_re", "ptbdb/patient001", source=VERI_KAYNAGI)
veri = pd.DataFrame({"QT": [196], "RR": [353]})
globals().pop("cevap", None)

def kisa_promptun_qtc_kodu(qt_samples, rr_samples, fs=360):
    qt_ms = qt_samples / fs * 1000
    rr_ms = rr_samples / fs * 1000
    return qt_ms / rr_ms ** 0.5


print("Kısa promptun kodunun verdiği QTc:", round(kisa_promptun_qtc_kodu(196, 353), 1), "ms")
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Prompt 3.4 kontrolü
teach.check_step("Prompt 3.4", "cevap", globals())

In [ ]:
#@title Prompt 3.4 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["ann_samples", "ann_symbols", "fs_mit", "fs_ptb", "header", "mit", "notes", "ptb", "read_physionet", "read_physionet_annotation"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    from vclinic.datasets import read_physionet, read_physionet_annotation
    mit, fs_mit, notes = read_physionet("100", "mitdb", source=VERI_KAYNAGI)
    ann_samples, ann_symbols = read_physionet_annotation("100", "mitdb", source=VERI_KAYNAGI)
    ptb, fs_ptb, header = read_physionet("s0010_re", "ptbdb/patient001", source=VERI_KAYNAGI)
veri = pd.DataFrame({"QT": [196], "RR": [353]})
fs = 500
qt_ms = veri["QT"].iloc[0] / fs * 1000      # milisaniye cinsinden QT
rr_s = veri["RR"].iloc[0] / fs              # saniye cinsinden RR aralığı
assert abs((200 / fs * 1000) / (500 / fs) ** 0.5 - 400) < 1e-9   # cevabı bilinen test
cevap = round(qt_ms / rr_s ** 0.5, 1)
print("Düzeltilmiş QT (ms):", cevap)
teach.check_step("Prompt 3.4", "cevap", globals())

**Sonuç ne gösteriyor?** İlk sonucu olanaksız kılan iki sessiz hata vardı: Asistan saniyede 360 örnek varsaymış ve RR aralığını karekökün altına milisaniye olarak koymuştu. Birimleri belirten Veri alanı, örnekleme frekansının varsayılmasını yasaklayan kısıt ve cevabı bilinen test, kod bir hastaya uygulanmadan önce bu iki hatayı önler.

**«Durak 3 · Prompt ile uygulama» tamamlandı.** Sırada hemen aşağıdaki «Durak 3 · Hazır uygulama» bölümü var. Durağı burada bırakacaksanız [ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#the-paper-of-the-week) ve okumaya «Haftanın makalesi» bölümünden devam edin.

### Durak 3 · Hazır uygulama

Bu bölüm gerçek kayıtlarla çalışır: MIT-BIH 100 numaralı kayıt üç kaynaktan biriyle okunur ve wfdb ile işlenir, atımları üç algoritmayla bulunur, kalp hızı değişkenliği hesaplanır, PTB veri tabanından gerçek bir 12 derivasyonlu EKG standart düzende çizilir ve kod güvenlik kontrol listesi bir YZ asistanının koduna uygulanır.

Hücreleri yukarıdan aşağıya çalıştırın. *Python notu* hücreleri kodu açıklar, *Gözlemle ve düşün* hücreleri ise sonuçlar üzerine sorular sorar.

In [ ]:
#@title Hazır uygulamanın hazırlığı: Bu hücreyi bir kez çalıştırın (0. bölüm ve uygulamanın ihtiyaç duyduğu önceki kodlar bu oturumda çalışmadıysa onları çalıştırır)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals() or any(_n not in globals() for _n in ["bandpass", "case", "detect_r_peaks", "measure_ecg"]):
    _catch_up(239, "0. bölüm ve burada gereken önceki kodlar çalıştı.")
print("3. durağın hazır uygulaması hazır.")

#### PhysioNet ve MIT-BIH'e erişim, wfdb ile okuma ve işleme, NeuroKit2 ile R-peak tespiti: 100 numaralı kayıt

*Ders: [PhysioNet ve MIT-BIH veri setlerine erişim](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#real-data-physionet-records) · [Wfdb kütüphanesi ile EKG okuma ve işleme](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#reading-and-processing-ecgs-with-the-wfdb-library) · [R-peak tespiti (NeuroKit2)](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#finding-every-beat-r-peak-detection)*

**Formüllerden koda.** Toplama penceresinin genişliği $N = 0{,}150 \cdot f_s$, `int(round(0.150 * fs))` ile hesaplanır ve 360 Hz'de 54 örnektir. Duyarlılık $Se = TP / (TP + FN)$ ve pozitif öngörü değeri $PPV = TP / (TP + FP)$, `vclinic.measure.match_beats` fonksiyonunun döndürdüğü sayılardan hesaplanır. Bulunan bir atım, referans atıma 150 ms'den daha yakınsa doğru sayılır.

Massachusetts Institute of Technology ve Beth Israel Hospital (MIT-BIH) Aritmi Veri Tabanı'nın 100 numaralı kaydı 30 dakika sürer ve 360 Hz'de kaydedilmiş iki kanal içerir. Kardiyologlar her atımı etiketlemiştir. Eğitim kaydın bir kopyasını GitHub deposunda tutar; veri hücresi kaydı PhysioNet'ten, bu kopyadan ya da bilgisayarınızdaki dosyalardan okur.

In [ ]:
# MIT-BIH 100 numaralı kaydı üç kaynaktan biriyle okuma
VERI_KAYNAGI = "web"   #@param ["kütüphane", "web", "bilgisayar"]

import urllib.request

import wfdb

RECORD = "100"
FILES = [RECORD + ".hea", RECORD + ".dat", RECORD + ".atr"]   # başlık, sinyaller ve atım etiketleri
URL = vclinic.COURSE_RAW + "data/physionet/mitdb/"   # PhysioNet dosyalarının ders deposundaki kopyası

if VERI_KAYNAGI == "kütüphane":
    # wfdb kaydı doğrudan PhysioNet'ten okur; pn_dir veri tabanının klasörüdür
    record = wfdb.rdrecord(RECORD, pn_dir="mitdb")
    annotation = wfdb.rdann(RECORD, "atr", pn_dir="mitdb")
else:
    if VERI_KAYNAGI == "web":
        # dosyalar web adresinden bu Jupyter Notebook'un klasörüne indirilir
        for name in FILES:
            urllib.request.urlretrieve(URL + name, name)
    else:
        # bilgisayardaki dosyalar: Dosyaları önce bu hücrenin altındaki nottaki bağlantıyla indirin
        vclinic.local_files(*FILES)   # Jupyter bu Jupyter Notebook'un klasörüne bakar; Colab bir yükleme penceresi açar
    record = wfdb.rdrecord(RECORD)
    annotation = wfdb.rdann(RECORD, "atr")

from vclinic.measure import match_beats

fs_mit = float(record.fs)
mit = pd.DataFrame(record.p_signal, columns=record.sig_name)
mit.index = pd.Index(np.round(np.arange(len(mit)) / fs_mit, 6), name="time_s")
ann_samples, ann_symbols = np.asarray(annotation.sample), list(annotation.symbol)
print("kaynak:", VERI_KAYNAGI, "| fs:", fs_mit, "| sinyaller:", record.sig_name, "| etiketler:", len(ann_samples))
print(pd.Series(ann_symbols).value_counts())
mit.head()

**Python notu.** Bir PhysioNet kaydı aynı adı taşıyan birkaç dosyadan oluşur: `100.hea` başlık dosyası sinyalleri tanımlar, `100.dat` sinyalleri, `100.atr` ise atım etiketlerini tutar. `wfdb.rdrecord` başlığı ve sinyalleri tek bir nesneye, `wfdb.rdann` ise etiketleri okur; `pn_dir` verildiğinde ikisi de doğrudan PhysioNet'ten okur. Son satırlar nesneyi, indeksi saniye cinsinden zaman olan `mit` tablosuna çevirir.

**Veriyi kendi bilgisayarınızdan okumak.** `"bilgisayar"` kaynağı için önce [mitdb_record_100.zip](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/raw/main/data/downloads/mitdb_record_100.zip) dosyasını indirin. Zip dosyasını açın ve içindeki `100.hea`, `100.dat` ve `100.atr` dosyalarını bu Jupyter Notebook'la aynı klasöre koyun. Ardından kaynak olarak `"bilgisayar"` seçin ve veri hücresini yeniden çalıştırın. Google Colab'da hücre, dosyaları bilgisayarınızdan seçmeniz için bir düğme gösterir; dosyaları birlikte seçin. Zip dosyasında verinin atıf bilgisi ve lisansı da yer alır.

Etiket sembolleri PhysioNet geleneğini izler: `N` normal atımı, `A` atriyal erken atımı, `V` ventriküler erken atımı gösterir. `+` bir atım değildir; ritim değişikliğini işaretler. `wfdb` paketi kuruluysa kaydı o okur; kurulu değilse `vclinic.physionet` içindeki küçük okuyucu aynı işi yapar.

**wfdb ile okuma.** `wfdb.rdrecord` bir kaydın başlık ve sinyal dosyalarını, `wfdb.rdann` ise etiket dosyasını okur. Kayıt örnekleme frekansını `fs`, sinyallerin adlarını `sig_name`, birimlerini `units`, milivolt cinsinden sinyalleri ise `p_signal` içinde tutar. `sampto` ile yalnızca ilk örnekler, burada ilk on saniye okunur ve `wfdb.plot_wfdb` bu kısmı etiketlerin sembolleriyle birlikte çizer.

In [ ]:
#@title Ders sayfasından mı geldiniz? Önce bu hücreyi çalıştırın (aşağıdaki kodun ihtiyaç duyduğu önceki kodları çalıştırır)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals() or any(_n not in globals() for _n in ["RECORD", "VERI_KAYNAGI", "annotation", "record", "wfdb"]):
    _catch_up(246, "0. bölüm ve burada gereken önceki kodlar çalıştı.")
print("Hazır: Aşağıdaki hücreleri çalıştırın.")

In [ ]:
# wfdb ile 100 numaralı kayıt: Başlık bilgileri ve ilk on saniye
print("fs:", record.fs, "| sinyaller:", record.sig_name, "| birimler:", record.units)
print("sinyal başına örnek:", record.sig_len, "| başlık yorumları:", record.comments)
print("etiketler:", len(annotation.sample), "| ilk semboller:", annotation.symbol[:8])
n10 = int(10 * record.fs)
if VERI_KAYNAGI == "kütüphane":
    first10 = wfdb.rdrecord(RECORD, pn_dir="mitdb", sampto=n10)
    ann10 = wfdb.rdann(RECORD, "atr", pn_dir="mitdb", sampto=n10)
else:
    first10 = wfdb.rdrecord(RECORD, sampto=n10)
    ann10 = wfdb.rdann(RECORD, "atr", sampto=n10)
fig = wfdb.plot_wfdb(record=first10, annotation=ann10, plot_sym=True, time_units="seconds",
                     title="MIT-BIH 100", figsize=(11, 5), return_fig=True)
for ax, name in zip(fig.axes, first10.sig_name):
    ax.set_ylabel(name + " (mV)")
fig.axes[-1].set_xlabel("Zaman (s)")
plt.show()

**wfdb ile işleme.** `processing.resample_singlechan` bir sinyali yeni bir frekansa yeniden örnekler ve etiketleri de yeni örneklere taşır. `processing.xqrs_detect` ise QRS komplekslerini paketin kendi algoritması XQRS ile bulur.

In [ ]:
# wfdb ile işleme: 250 Hz'e yeniden örnekleme ve XQRS algoritması
try:
    from wfdb import processing

    x360 = record.p_signal[:, 0]
    x250, ann250 = processing.resample_singlechan(x360, annotation, record.fs, 250)
    print("360 Hz'deki örnek sayısı:", len(x360), "-> 250 Hz'de:", len(x250))
    print("ilk etiketin örnek numarası:", annotation.sample[0], "->", ann250.sample[0])
    xqrs_peaks = processing.xqrs_detect(x360, fs=record.fs, verbose=False)
    print("XQRS:", len(xqrs_peaks), "QRS kompleksi")
except (ImportError, NameError):
    xqrs_peaks = None
    print("wfdb kurulu değil; bu adım atlanıyor.")

↩ *[Ders sayfasında kaldığınız yere dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#nbref-c212)*

In [ ]:
# Algoritmanın dört aşaması, ilk 3 saniyede
x = mit["MLII"].to_numpy()
seg = slice(0, int(3 * fs_mit))
y = bandpass(x, fs_mit, 5, 15)
z = (np.gradient(y) * fs_mit) ** 2
w = int(round(0.150 * fs_mit))
m = np.convolve(z, np.ones(w) / w, mode="same")
t = np.arange(len(x)) / fs_mit
fig, axes = plt.subplots(4, 1, figsize=(11, 7), sharex=True)
names = ["ham MLII derivasyonu (mV)", "bant geçiren 5-15 Hz",
         "eğimin karesi", f"hareketli pencere ({w} örnek)"]
for ax, sig, name in zip(axes, [x, y, z, m], names):
    ax.plot(t[seg], sig[seg], color="#9B1B1B" if ax is axes[3] else "k", lw=0.9)
    ax.set_ylabel(name, fontsize=8)
axes[3].set_xlabel("Zaman (s)")
plt.tight_layout()
plt.show()

In [ ]:
# Tüm kayıtta tespit ve etiketlerle karşılaştırma
x = mit["MLII"].to_numpy()
beat_symbols = set("NLRBAaJSVrFejnE/fQ?")
ref = np.array([s for s, sym in zip(ann_samples, ann_symbols) if sym in beat_symbols])
found = detect_r_peaks(x, fs_mit)
tp, fp, fn = match_beats(found / fs_mit, ref / fs_mit, window_s=0.150)
print(f"referans atımlar: {len(ref)}, bulunan: {len(found)}")
print(f"TP = {tp}, FP = {fp}, FN = {fn}")
print(f"Se = {100 * tp / (tp + fn):.2f} %, PPV = {100 * tp / (tp + fp):.2f} %")

**NeuroKit2 ile R-peak tespiti.** `nk.ecg_clean` sinyali temizler (varsayılan olarak 0,5 Hz'lik yüksek geçiren filtre ve şebeke elektriği için bir filtre kullanır), `nk.ecg_peaks` ise R tepelerini varsayılan yöntemiyle bulur. Tablo üç algoritmayı aynı etiketli atımlar üzerinde karşılaştırır: Dersin algoritması, wfdb'nin XQRS algoritması ve NeuroKit2.

In [ ]:
#@title Ders sayfasından mı geldiniz? Önce bu hücreyi çalıştırın (aşağıdaki kodun ihtiyaç duyduğu önceki kodları çalıştırır)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals() or any(_n not in globals() for _n in ["found", "fs_mit", "match_beats", "ref", "x", "xqrs_peaks"]):
    _catch_up(254, "0. bölüm ve burada gereken önceki kodlar çalıştı.")
print("Hazır: Aşağıdaki hücreleri çalıştırın.")

In [ ]:
# NeuroKit2 ile R-peak tespiti ve üç algoritmanın karşılaştırılması
rows = [("dersin algoritması", found)]
if globals().get("xqrs_peaks") is not None:
    rows.append(("XQRS (wfdb)", np.asarray(xqrs_peaks)))
try:
    import neurokit2 as nk

    cleaned = nk.ecg_clean(x, sampling_rate=fs_mit)
    _, info = nk.ecg_peaks(cleaned, sampling_rate=fs_mit)
    nk_peaks = np.asarray(info["ECG_R_Peaks"])
    rows.append(("NeuroKit2", nk_peaks))
except ImportError:
    nk_peaks = None
    print("neurokit2 kurulu değil; pip install neurokit2 komutuyla kurulabilir.")
table = []
for name, peaks in rows:
    tp, fp, fn = match_beats(peaks / fs_mit, ref / fs_mit, window_s=0.150)
    table.append({"algoritma": name, "bulunan": len(peaks), "TP": tp, "FP": fp, "FN": fn,
                  "Se (%)": round(100 * tp / (tp + fn), 2), "PPV (%)": round(100 * tp / (tp + fp), 2)})
pd.DataFrame(table)

↩ *[Ders sayfasında kaldığınız yere dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#nbref-c218)*

**Gözlemle ve düşün.** Algoritma 100 numaralı kaydın bütün atımlarını buluyor. Bu sonuç, algoritmaya her kayıtta güvenmek için neden yeterli değildir? Gürültüyü, geniş ventriküler atımları ve yüksek T dalgalarını düşünün. Algoritmayı zorlamak için bir test kümesine hangi kayıtları koyardınız?

**İleri düzey (isteğe bağlı).** İnternet bağlantısı varsa `wfdb.rdrecord("105", pn_dir="mitdb")` ve `wfdb.rdann("105", "atr", pn_dir="mitdb")`, gürültüsüyle bilinen 105 numaralı kaydı doğrudan PhysioNet'ten okur. Algoritmayı bu kayıtta çalıştırın; neurokit2 kuruluysa sonucu `neurokit2.ecg_peaks` ile karşılaştırın.

In [ ]:
# İleri düzey (isteğe bağlı): PhysioNet'ten 105 numaralı kayıt (internet ve wfdb gerekir)
try:
    import wfdb
    rec = wfdb.rdrecord("105", pn_dir="mitdb", sampto=360 * 300)
    ann = wfdb.rdann("105", "atr", pn_dir="mitdb", sampto=360 * 300)
    ref105 = np.array([s for s, sym in zip(ann.sample, ann.symbol) if sym in beat_symbols])
    found105 = detect_r_peaks(rec.p_signal[:, 0], rec.fs)
    print(match_beats(found105 / rec.fs, ref105 / rec.fs))
except Exception as err:
    print("Burada kullanılamıyor:", type(err).__name__)

#### HRV (kalp hızı değişkenliği) hesaplama

*Ders: [HRV (kalp hızı değişkenliği) hesaplama](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#heart-rate-variability)*

**Formüllerden koda.** RR aralıkları milisaniye cinsinden bir `rr_ms` dizisindeyse NN aralıklarının standart sapması $SDNN = \sqrt{\frac{1}{N-1}\sum (RR_i - \overline{RR})^2}$, `np.std(rr_ms, ddof=1)` ile hesaplanır. Ardışık farkların karelerinin ortalamasının karekökü $RMSSD = \sqrt{\frac{1}{N-1}\sum (RR_{i+1} - RR_i)^2}$ ise `np.sqrt(np.mean(np.diff(rr_ms) ** 2))` ile bulunur.

In [ ]:
# Dersteki çözümlü örnek: Altı RR aralığı (ms)
rr_ms = np.array([812, 845, 830, 798, 860, 841])
print("ortalama RR:", rr_ms.mean())
print("SDNN:", round(np.std(rr_ms, ddof=1), 2), "ms")
print("ardışık farklar:", np.diff(rr_ms))
print("RMSSD:", round(np.sqrt(np.mean(np.diff(rr_ms) ** 2)), 2), "ms")

In [ ]:
# Vaka için beş dakikalık D2 kaydı; 1996 Görev Grubu kısa süreli HRV için beş dakikayı önermektedir
from vclinic.ecg import lead_ii_long

t5, x5, truth5 = lead_ii_long(case.params, duration_s=300, fs=250, seed=17)
r5 = detect_r_peaks(x5, 250)
rr5 = np.diff(r5) / 250 * 1000                 # ms cinsinden RR aralıkları
sdnn = np.std(rr5, ddof=1)
rmssd = np.sqrt(np.mean(np.diff(rr5) ** 2))
print(f"atım: {len(r5)}, ortalama kalp hızı: {60000 / rr5.mean():.1f} atım/dk, SDNN: {sdnn:.1f} ms, RMSSD: {rmssd:.1f} ms")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(r5[1:] / 250, rr5, color="#9B1B1B", lw=0.8)
axes[0].set_xlabel("Zaman (s)")
axes[0].set_ylabel("RR (ms)")
axes[0].set_title("Takogram")
axes[1].scatter(rr5[:-1], rr5[1:], s=6, color="#1F5F8B")
axes[1].set_xlabel("RR$_i$ (ms)")
axes[1].set_ylabel("RR$_{i+1}$ (ms)")
axes[1].set_title("Poincaré grafiği")
plt.tight_layout()
plt.show()

In [ ]:
# 100 numaralı kayıt: İlk 5 dakikanın NN aralıkları
pairs = [(s1, s2) for (s1, a), (s2, b) in zip(zip(ann_samples, ann_symbols),
                                               zip(ann_samples[1:], ann_symbols[1:]))
         if a == "N" and b == "N" and s2 <= 300 * fs_mit]
nn_ms = np.array([(s2 - s1) / fs_mit * 1000 for s1, s2 in pairs])
print(f"NN aralık: {len(nn_ms)}, ortalama kalp hızı: {60000 / nn_ms.mean():.1f} atım/dk")
print(f"SDNN: {np.std(nn_ms, ddof=1):.1f} ms, RMSSD: {np.sqrt(np.mean(np.diff(nn_ms) ** 2)):.1f} ms")

**NeuroKit2 ile HRV.** `nk.hrv_time`, SDNN, RMSSD ve daha birçok ölçütü R tepelerinden ya da milisaniye cinsinden RR aralıklarından ve zamanlarından hesaplar. Tablo, ilk beş dakikada bulunan bütün atımlar için elde edilen sonucu, yalnızca NN aralıklarıyla elde edilen sonuçla ve elle yapılan hesapla karşılaştırır.

In [ ]:
#@title Ders sayfasından mı geldiniz? Önce bu hücreyi çalıştırın (aşağıdaki kodun ihtiyaç duyduğu önceki kodları çalıştırır)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals() or any(_n not in globals() for _n in ["fs_mit", "mit", "nn_ms", "pairs"]):
    _catch_up(265, "0. bölüm ve burada gereken önceki kodlar çalıştı.")
print("Hazır: Aşağıdaki hücreleri çalıştırın.")

In [ ]:
# NeuroKit2 ile HRV: Bulunan bütün atımlar ve NN aralıkları
try:
    import neurokit2 as nk

    peaks_nk = globals().get("nk_peaks")
    if peaks_nk is None:
        _, info = nk.ecg_peaks(nk.ecg_clean(mit["MLII"].to_numpy(), sampling_rate=fs_mit), sampling_rate=fs_mit)
        peaks_nk = np.asarray(info["ECG_R_Peaks"])
    all_beats = nk.hrv_time(peaks_nk[peaks_nk <= 300 * fs_mit], sampling_rate=fs_mit)
    nn_times = np.array([s2 / fs_mit for s1, s2 in pairs])
    nn_only = nk.hrv_time({"RRI": nn_ms, "RRI_Time": nn_times})
    summary = pd.DataFrame({
        "SDNN (ms)": [all_beats["HRV_SDNN"].iloc[0], nn_only["HRV_SDNN"].iloc[0], np.std(nn_ms, ddof=1)],
        "RMSSD (ms)": [all_beats["HRV_RMSSD"].iloc[0], nn_only["HRV_RMSSD"].iloc[0], np.sqrt(np.mean(np.diff(nn_ms) ** 2))]},
        index=["NeuroKit2, bulunan bütün atımlar",
               "NeuroKit2, NN aralıkları",
               "elle, NN aralıkları"]).round(1)
    display(summary)
except ImportError:
    print("neurokit2 kurulu değil; pip install neurokit2 komutuyla kurulabilir.")

↩ *[Ders sayfasında kaldığınız yere dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#nbref-c232)*

**Gözlemle ve düşün.** Bulunan bütün atımlardan hesaplanan RMSSD, NN aralıklarından hesaplanan RMSSD'nin yaklaşık iki katıdır. İlk beş dakikadaki hangi atımlar bu farka yol açar ve kısa süreli HRV'nin tanımı neden yalnızca NN aralıklarını kullanır?

**Keşfet.** Sanal kliniğin sinyal üreticisinde `hrv` adlı bir değişkenlik parametresi vardır: 1,0 değeri istirahatteki tipik bir yetişkini temsil eder; daha küçük değerler ileri yaşı ya da kalp yetersizliğini taklit eder. Parametreyi değiştirin ve beş dakikalık bir kaydın SDNN ve RMSSD değerlerini izleyin.

In [ ]:
# Keşfet: değişkenlik ve iki ölçüt
def show_hrv(hrv=1.0, heart_rate=70):
    params = dict(case.params, hrv=hrv, hr_bpm=heart_rate)
    _, xs, _ = lead_ii_long(params, duration_s=300, fs=250, seed=5)
    rr = np.diff(detect_r_peaks(xs, 250)) / 250 * 1000
    print(f"hrv = {hrv:.2f}, HR = {heart_rate} atım/dk -> SDNN = {np.std(rr, ddof=1):.1f} ms, "
          f"RMSSD = {np.sqrt(np.mean(np.diff(rr) ** 2)):.1f} ms")


try:
    from ipywidgets import interact, FloatSlider, IntSlider
    interact(show_hrv, hrv=FloatSlider(value=1.0, min=0.2, max=1.6, step=0.1),
             heart_rate=IntSlider(value=70, min=50, max=110, step=5, description="HR"))
except ImportError:
    show_hrv(1.0, 70)
    show_hrv(0.4, 70)

**Gözlemle ve düşün.** RMSSD daha çok atımdan atıma olan hızlı değişimlere, SDNN ise yavaş değişimlere de duyarlıdır. 1996 Görev Grubu kısa süreli ölçütler için neden 5 dakikalık kayıtları önermektedir ve farklı uzunluktaki kayıtlardan elde edilen değerler neden karşılaştırılmamalıdır?

#### 12 derivasyon görselleştirme: PTB Tanısal EKG Veri Tabanı'ndan gerçek bir 12 derivasyonlu EKG

*Ders: [12 derivasyon görselleştirme](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#visualising-the-12-leads)*

Almanya'daki Physikalisch-Technische Bundesanstalt (PTB) kurumunun PTB Tanısal EKG Veri Tabanı'nda 1 numaralı hastaya ait `s0010_re` kaydı, 1.000 Hz'de alınmış 38,4 saniyelik 12 standart derivasyon ve üç Frank derivasyonu içerir. Kaydın başlık dosyasında hastanın klinik özeti bulunur. Bölüm 12 derivasyonu iki kez çizer: Önce alt alta, sonra klinikteki standart çıktı düzeninde. Bu kayıt, «Durak 2 · Hazır uygulama» bölümündeki ölçüm adımlarının gerçek bir kayıtta nasıl çalıştığını da göstermektedir.

In [ ]:
# PTB veri tabanındaki 1 numaralı hastanın s0010_re kaydını üç kaynaktan biriyle okuma
VERI_KAYNAGI = "web"   #@param ["kütüphane", "web", "bilgisayar"]

import urllib.request

import wfdb

PTB_RECORD = "s0010_re"
PTB_FILES = [PTB_RECORD + ".hea", PTB_RECORD + ".dat", PTB_RECORD + ".xyz"]   # başlık, 12 derivasyon ve 3 Frank derivasyonu
PTB_URL = vclinic.COURSE_RAW + "data/physionet/ptbdb/patient001/"   # PhysioNet dosyalarının ders deposundaki kopyası

if VERI_KAYNAGI == "kütüphane":
    # wfdb kaydı doğrudan PhysioNet'ten okur; pn_dir veri tabanının klasörüdür
    record_ptb = wfdb.rdrecord(PTB_RECORD, pn_dir="ptbdb/patient001")
else:
    if VERI_KAYNAGI == "web":
        # dosyalar web adresinden bu Jupyter Notebook'un klasörüne indirilir
        for name in PTB_FILES:
            urllib.request.urlretrieve(PTB_URL + name, name)
    else:
        # bilgisayardaki dosyalar: Dosyaları önce bu hücrenin altındaki nottaki bağlantıyla indirin
        vclinic.local_files(*PTB_FILES)   # Jupyter bu Jupyter Notebook'un klasörüne bakar; Colab bir yükleme penceresi açar
    record_ptb = wfdb.rdrecord(PTB_RECORD)

fs_ptb = float(record_ptb.fs)
ptb = pd.DataFrame(record_ptb.p_signal, columns=record_ptb.sig_name)
ptb.index = pd.Index(np.round(np.arange(len(ptb)) / fs_ptb, 6), name="time_s")
header = list(record_ptb.comments)
print("kaynak:", VERI_KAYNAGI, "|", ptb.shape, fs_ptb)
for line in header[:9]:
    print(line)

**Veriyi kendi bilgisayarınızdan okumak.** `"bilgisayar"` kaynağı için önce [ptbdb_s0010_re.zip](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/raw/main/data/downloads/ptbdb_s0010_re.zip) dosyasını indirin. Zip dosyasını açın ve içindeki `s0010_re.hea`, `s0010_re.dat` ve `s0010_re.xyz` dosyalarını bu Jupyter Notebook'la aynı klasöre koyun. Ardından kaynak olarak `"bilgisayar"` seçin ve veri hücresini yeniden çalıştırın. Google Colab'da hücre, dosyaları bilgisayarınızdan seçmeniz için bir düğme gösterir; dosyaları birlikte seçin. Zip dosyasında verinin atıf bilgisi ve lisansı da yer alır.

In [ ]:
# Gerçek kaydın on iki derivasyonu (sütun adları küçük harfle yazılmıştır)
names = {"I": "i", "II": "ii", "III": "iii", "aVR": "avr", "aVL": "avl", "aVF": "avf",
         "V1": "v1", "V2": "v2", "V3": "v3", "V4": "v4", "V5": "v5", "V6": "v6"}
fig = teach.plot_12lead(ptb.index.to_numpy(), ptb, title="PTB patient001, s0010_re",
                        v_range=(-1.5, 1.5), names=names)
plt.show()

**Standart çıktı.** 12 derivasyonlu EKG dört sütunlu üç satır hâlinde basılır: D1, D2 ve D3 birinci sütunda, aVR, aVL ve aVF ikinci sütunda, V1-V3 üçüncü sütunda, V4-V6 ise dördüncü sütunda yer alır. Her derivasyon 2,5 saniyelik bir alan kaplar ve dördüncü satır, ritim şeridi olarak 10 saniyenin tamamında D2'yi gösterir. Kâğıtta 25 mm/s ve 10 mm/mV ayarıyla küçük bir kare 0,04 s genişliğinde ve 0,1 mV yüksekliğindedir; beş küçük kareden oluşan büyük bir kare ise 0,2 s genişliğinde ve 0,5 mV yüksekliğindedir.

**Python notu.** `plt.subplots(4, 1)` dört satırlık eksen takımı oluşturur. `(t >= 2.5 * col) & (t < 2.5 * (col + 1))` bir Boolean maskesidir: 2,5 saniyelik bir sütunun örneklerinde doğru değerini alır. `ax.set_aspect(0.4)` kâğıdın oranını korur, çünkü 1 mV 10 mm, 1 s ise 25 mm olarak çizilir ve 10 / 25 = 0,4'tür.

In [ ]:
# Standart çıktı: 25 mm/s ve 10 mm/mV ile D2 ritim şeritli 3 × 4 düzen
layout = [["I", "aVR", "V1", "V4"], ["II", "aVL", "V2", "V5"], ["III", "aVF", "V3", "V6"]]
shown = {"I": "D1", "II": "D2", "III": "D3"}
ten = ptb.iloc[: int(10 * fs_ptb)]                               # ilk 10 saniye
t = ten.index.to_numpy()
fig, axes = plt.subplots(4, 1, figsize=(12, 7.6))
for row, leads_row in enumerate(layout):
    for col, lead in enumerate(leads_row):
        part = (t >= 2.5 * col) & (t < 2.5 * (col + 1))         # her derivasyon 2,5 s yer kaplar
        axes[row].plot(t[part], ten[names[lead]].to_numpy()[part], color="k", lw=0.8)
        axes[row].text(2.5 * col + 0.06, 1.62, shown.get(lead, lead), fontsize=9)
axes[3].plot(t, ten["ii"].to_numpy(), color="k", lw=0.8)         # ritim şeridi
axes[3].text(0.06, 1.62, shown["II"], fontsize=9)
for ax in axes:
    ax.set_xlim(0, 10)
    ax.set_ylim(-1.5, 2.0)
    ax.set_xticks(np.arange(0, 10.001, 0.2))                      # büyük kare: 0,2 s
    ax.set_xticks(np.arange(0, 10.001, 0.04), minor=True)         # küçük kare: 0,04 s
    ax.set_yticks(np.arange(-1.5, 2.001, 0.5))                    # büyük kare: 0,5 mV
    ax.set_yticks(np.arange(-1.5, 2.001, 0.1), minor=True)        # küçük kare: 0,1 mV
    ax.grid(which="major", color="#E8A0A0", lw=0.7)
    ax.grid(which="minor", color="#F6DADA", lw=0.4)
    ax.tick_params(which="both", labelbottom=False, labelleft=False, length=0)
    ax.set_aspect(0.4)                                            # 1 mV = 10 mm ve 1 s = 25 mm
axes[3].set_xlabel("25 mm/s ve 10 mm/mV ile 10 s")
plt.tight_layout()
plt.show()

In [ ]:
# Gerçek veride Einthoven ve Goldberger ilişkileri
print("max |II - (I + III)| =", round(float(np.abs(ptb["ii"] - (ptb["i"] + ptb["iii"])).max()), 4), "mV")
print("max |aVR + aVL + aVF| =", round(float(np.abs(ptb["avr"] + ptb["avl"] + ptb["avf"]).max()), 4), "mV")

In [ ]:
# Bölüm 5'teki ölçüm gerçek kayıtta
leads_ptb = ptb[[names[k] for k in vclinic.LEADS]].to_numpy()
m_ptb = measure_ecg(leads_ptb, fs_ptb)
pd.Series(m_ptb).round(1)

**Gözlemle ve düşün.** Algoritma bu kayıtta yaklaşık 137 ms'lik bir QRS süresi ölçüyor. Şekildeki V1 derivasyonuna ve inferior derivasyonlara bakın. Bu genişlikte bir QRS'yi hangi örüntü açıklayabilir ve inferior derivasyonlar ne gösteriyor? 2009 önerileri bilgisayar ölçümlerinin gözle denetlenmesini istemektedir: Şekil üzerinde bir atımı kendiniz ölçün ve karşılaştırın.

**İleri düzey (isteğe bağlı).** `vx`, `vy` ve `vz` Frank derivasyonları kalp vektörünü neredeyse doğrudan kaydeder. Bir atımın QRS halkasını üç boyutta çizin. Ders kitapları Frank derivasyonları için farklı işaret kuralları kullandığından halka burada kaydedildiği gibi çizilmiştir.

In [ ]:
# İleri düzey (isteğe bağlı): Frank derivasyonlarından bir atımın QRS halkası
r_ptb = detect_r_peaks(ptb["ii"].to_numpy(), fs_ptb)
k = r_ptb[3]
loop = ptb[["vx", "vy", "vz"]].to_numpy()[k - int(0.06 * fs_ptb):k + int(0.08 * fs_ptb)]
fig = plt.figure(figsize=(5.5, 5))
ax = fig.add_subplot(projection="3d")
ax.plot(loop[:, 0], loop[:, 1], loop[:, 2], color="#9B1B1B")
ax.set_xlabel("vx (mV)")
ax.set_ylabel("vy (mV)")
ax.set_zlabel("vz (mV)")
plt.show()

#### YZ asistanıyla çalışmak: Kod güvenlik kontrol listesi

*Ders: [YZ asistanıyla çalışmak, 1. bölüm: Kod güvenlik kontrol listesi](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#ai-pair-programming-part-1-the-code-safety-checklist)*

Bir YZ asistanı bu eğitimdeki kodların çoğunu yazabilir; ama hatasız çalışıp yanlış bir sayı döndüren kod da yazabilir. Bu yüzden eğitimde YZ'nin yazdığı kod, ekibe yeni katılmış bir üyenin işi gibi ele alınır: Hoş karşılanır ve denetlenir. Aşağıdaki prompt kartı iyi bir promptun yapısını göstermektedir; bu kart, «Durak 3 · Prompt ile uygulama» bölümündeki Prompt 3.4'ün kartıdır.

**Prompt kartı.** *Rol:* Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım. *Bağlam:* Saniyede 500 kez örneklenmiş bir EKG'de QT ve RR aralıkları örnek sayısı olarak ölçüldü. Bazett yöntemiyle düzeltilmiş QT aralığını milisaniye olarak bulmak istiyorum. *Veri:* Jupyter Notebook'ta "veri" adında tek satırlık bir tablo var: "QT" sütunu QT aralığını, "RR" sütunu RR aralığını örnek sayısı olarak verir. Kayıt saniyede 500 kez örneklenmiştir. *Görev:* QT aralığını milisaniyeye, RR aralığını saniyeye çevir. QT'yi Bazett yöntemiyle düzelt: Milisaniye cinsinden QT'yi, saniye cinsinden RR aralığının kareköküne böl. *Kısıtlar:* Yalnızca bu tabloyu kullan. Saniyedeki örnek sayısını kendin varsayma; burada verilen 500 değerini kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır. *Çıktı:* Düzeltilmiş QT'yi milisaniye olarak, bir ondalık basamakla yazdır ve "cevap" adıyla sakla. *Test:* Aynı hesap, QT 200 örnek ve RR 500 örnek olduğunda tam 400 ms vermeli; çünkü bu durumda RR aralığı tam 1 saniyedir.

**Kullanmadan önce denetleyin.** Ders sayfasında eğitimin kod güvenlik kontrol listesi yer alır: Çalıştırmadan önce *giriş* (birimler, örnekleme frekansı, derivasyon ve hasta tanımlayıcısı bulunmaması), çalışırken *mola* (hatalar ve fizyolojik değerler) ve kullanmadan önce *çıkış* (bilinen bir gerçek değerle karşılaştırma, her satırın açıklanması, promptun ve değişikliklerin kaydı).

Aşağıdaki fonksiyonu bir YZ asistanı, örnekleme frekansını belirtmeyen daha kısa bir prompta bakarak yazmıştır.

In [ ]:
# YZ asistanının kısa bir prompta bakarak yazdığı kod
def qtc_from_samples(qt_samples, rr_samples, fs=360):
    qt_ms = qt_samples / fs * 1000
    rr_ms = rr_samples / fs * 1000
    return qt_ms / np.sqrt(rr_ms)


print(qtc_from_samples(196, 353))

**Gözlemle ve düşün.** Fonksiyon hatasız çalışıyor. Kontrol listesini adım adım uygulayın: Sonuçtaki sorunu hangi denetim noktasında fark edersiniz ve koddaki iki hata nedir? Hatalardan biri, promptun hiç istemediği varsayılan bir değerden kaynaklanmaktadır.

**YZ kullanım günlüğü.** Her haftalık projeye, bir YZ asistanının her kullanımı için bir satır içeren kısa bir günlük eklenir:

| Araç | Prompt (kısaca) | Neyin denetlendiği | Neyin değiştirildiği |
|---|---|---|---|
| (asistanın adı) | Örnek sayılarından Bazett QTc hesabı | 466,5 ms sonucunun VC-0317'nin gerçek değerleriyle karşılaştırılması | Varsayılan fs kaldırıldı; RR saniyeye çevrildi |

**Promptların sonuçları.** Aşağıdaki hücre 3. durağın promptlarının sonuçlarını bu uygulamanın koduyla hesaplar ve promptların kontrol hücreleri gibi denetler.

In [ ]:
# Bu durağın promptlarının sonuçları, Python ile
from scipy.signal import find_peaks

n_atrial = ann_symbols.count("A")
print("100 numaralı kayıttaki atriyal erken atımlar:", n_atrial)
teach.check("Prompt 3.1", n_atrial)
beats = [(s, b) for s, b in zip(ann_samples, ann_symbols) if b != "+"]   # "+" bir atımı değil ritim değişikliğini gösterir
kinds = np.array([b for _, b in beats])
rr_all = np.diff([s for s, _ in beats]) / fs_mit * 1000
nn_all = rr_all[(kinds[:-1] == "N") & (kinds[1:] == "N")]
sdnn_all = round(np.std(nn_all, ddof=1), 1)
print("Bütün kaydın SDNN değeri (ms):", sdnn_all)
teach.check("Prompt 3.2", sdnn_all)
lead_i = ptb["i"].to_numpy()
r_ptb, _ = find_peaks(lead_i, height=lead_i.max() / 2, distance=int(0.3 * fs_ptb))
hr_ptb = round(60 / np.diff(r_ptb / fs_ptb).mean(), 1)
print("PTB kaydının kalp hızı (atım/dk):", hr_ptb)
teach.check("Prompt 3.3", hr_ptb)
qtc_safe = round((196 / 500 * 1000) / (353 / 500) ** 0.5, 1)   # kod güvenlik kontrol listesinin düzelttiği hesap
print("500 Hz'de 196 ve 353 örnekten Bazett QTc'si (ms):", qtc_safe)
teach.check("Prompt 3.4", qtc_safe)

**3. durak tamamlandı.** [Ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#the-paper-of-the-week) ve okumaya «Haftanın makalesi» bölümünden devam edin. Haftalık proje bu Jupyter Notebook'un sonundadır; ders sayfasındaki «Haftalık proje» bölümü projeyi anlatır.

## Haftalık proje

*Ders sayfasında: [Haftalık proje](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#weekly-project) · [1. haftanın README sayfası, Haftalık proje](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-01/README.md#haftal%C4%B1k-proje)*

**Proje.** Bir EKG'yi çizen ve temel ölçümlerini (PR, QRS, QT) kendiliğinden hesaplayan bir script. Proje «Durak 2 · Hazır uygulama» bölümündeki `measure_ecg` fonksiyonundan başlar, frontal QRS eksenini ekler, ölçümleri on hastanın gerçek değerleriyle karşılaştırır, 100 numaralı kayıtta kalp hızını otuz dakika boyunca izler ve her hasta için Bazett ile Fridericia arasında seçim yapar.

Bu bölüm kendi başına çalışır: Hazırlık hücresini bir kez çalıştırın; hücre projenin ihtiyaç duyduğu her şeyi yükler. Proje, her adım için bir prompt olmak üzere aşağıdaki promptlarla yapılabilir. Her adım duraklardaki promptlar gibi ilerler; ancak sonuçlar rapora girdiği için kontrol hücresi ve hazır kod yoktur. Bir promptun testi sağlanmadan sonraki prompta geçilmez ve her prompt YZ kullanım günlüğüne yazılır. Python yazanlar bu bölümün sonunda bir başlangıç kodu bulur.

In [ ]:
#@title Projenin hazırlığı: Bu hücreyi bir kez çalıştırın (projenin verilerini yükler; 0. bölüm bu oturumda çalışmadıysa önce onu çalıştırır)
VERI_KAYNAGI = "web"   #@param ["kütüphane", "web", "bilgisayar"]
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
from vclinic import measure as _M
from vclinic.datasets import read_physionet_annotation
clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
ann_samples, ann_symbols = read_physionet_annotation("100", "mitdb")
fs_mit = 360
proje_hastalari = ["VC-0317", "VC-0079", "VC-0254", "VC-0188", "VC-0042",
                     "VC-0150", "VC-0007", "VC-0031", "VC-0018", "VC-0026"]


def _project_measure(e):
    """Heart rate, PR, QRS, QT and frontal axis of one ECG, with the net QRS voltages of leads I and aVF in every beat."""
    L, fs = e.leads.to_numpy(), e.fs
    r = _M.detect_r_peaks(L[:, 1], fs)
    on, off = _M.qrs_bounds(L, fs, r)
    x20 = _M.lowpass(L[:, 1], fs, 20)
    sinus = e.truth["rhythm"] == "sinus"
    pr, qrs, qt, net_i, net_avf, t_prev = [], [], [], [], [], None
    for k in range(len(r) - 1):
        b = max(0, on[k] - int(0.03 * fs))
        te = _M.t_end_tangent(x20, fs, off[k], r[k + 1], np.median(x20[b:on[k]]))
        qrs.append((off[k] - on[k]) / fs * 1000)
        qt.append((te - on[k]) / fs * 1000)
        if sinus and t_prev is not None and np.isfinite(t_prev):
            pr.append((on[k] - _M.p_onset(x20, fs, on[k], earliest=t_prev + 0.02 * fs)) / fs * 1000)
        t_prev = te
        for col, out in ((0, net_i), (5, net_avf)):
            seg = L[on[k]:off[k] + 1, col] - np.median(L[b:on[k], col])
            out.append(seg.max() + seg.min())
    axis = [vclinic.frontal_axis(a, v) for a, v in zip(net_i, net_avf)]
    return {"hr": 60 / (np.mean(np.diff(r)) / fs), "pr": np.nanmedian(pr) if np.isfinite(pr).any() else np.nan,
            "qrs": np.median(qrs), "qt": np.nanmedian(qt), "axis": float(np.median(axis)),
            "net_i": net_i, "net_avf": net_avf}


def _project_table(patients):
    rows = []
    for pid in patients:
        p = clinic.set_index("patient_id").loc[pid]
        e = vclinic.patient_ecg(clinic, pid)
        m, t = _project_measure(e), e.truth
        for name, key, true in [("kalp hızı", "hr", t["hr_bpm"]), ("PR", "pr", t["pr_ms"]), ("QRS", "qrs", t["qrs_ms"]),
                                ("QT", "qt", t["qt_ms"]), ("eksen", "axis", t["axis_deg"])]:
            rows.append({"hasta": pid, "ritim": "sinüs" if p["ecg_rhythm"] == "sinus" else "AF",
                         "ileti": p["ecg_conduction"], "ölçüm": name,
                         "ölçülen": round(float(m[key]), 1), "gerçek": round(float(true), 1)})
    return pd.DataFrame(rows)


def _project_qt(patients):
    rows = []
    for pid in patients:
        p = clinic.set_index("patient_id").loc[pid]
        m = _project_measure(vclinic.patient_ecg(clinic, pid))
        rows.append({"hasta": pid, "cinsiyet": "kadın" if p["sex"] == "F" else "erkek",
                     "kalp hızı": round(m["hr"], 1), "QT": round(m["qt"], 1)})
    return pd.DataFrame(rows)
print("Proje hazır.")

### Proje adımı 1: Vakanın frontal ekseni

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: 67 yaşında bir kadın hastanın istirahat EKG'sinden frontal QRS eksenini atım atım ölçmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var. Her satır bir atımdır: "atım" sütunu atımın sırasını, "D1" ve "aVF" sütunları da bu derivasyonlardaki net QRS gerilimini milivolt olarak verir. Net gerilim, QRS kompleksindeki en büyük pozitif ve en büyük negatif sapmanın toplamıdır.  
> Görev: Her atım için yatay bileşeni D1 gerilimi, dikey bileşeni aVF geriliminin 0,866'ya bölümü olan vektörü kur. Frontal eksen, bu vektörün derece cinsinden açısıdır; açı D1 yönünden ölçülür ve ayaklara doğru artı değer alır. Ardından bütün atımların medyan eksenini bul ve −30 ile +90 derece arasındaki normal aralıkta olup olmadığını yazdır.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Medyan ekseni derece olarak, bir ondalık basamakla yazdır ve "cevap" adıyla sakla.  
> Test: D1'de 0,9 mV, aVF'de −0,3 mV olan bir atım yaklaşık −21 derecelik bir eksen vermeli. Kayıtta bu hastanın ekseni 21 derecedir; sonuç bu değerin yaklaşık 10 derece yakınında olmalı, çünkü net gerilimler küçük septal ve terminal kuvvetleri de içerir.  

In [ ]:
#@title Proje adımı 1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["ann_samples", "ann_symbols", "clinic", "fs_mit", "project_patients", "read_physionet_annotation"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    from vclinic import measure as _M
    from vclinic.datasets import read_physionet_annotation
    clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
    ann_samples, ann_symbols = read_physionet_annotation("100", "mitdb")
    fs_mit = 360
    proje_hastalari = ["VC-0317", "VC-0079", "VC-0254", "VC-0188", "VC-0042",
                         "VC-0150", "VC-0007", "VC-0031", "VC-0018", "VC-0026"]


    def _project_measure(e):
        """Heart rate, PR, QRS, QT and frontal axis of one ECG, with the net QRS voltages of leads I and aVF in every beat."""
        L, fs = e.leads.to_numpy(), e.fs
        r = _M.detect_r_peaks(L[:, 1], fs)
        on, off = _M.qrs_bounds(L, fs, r)
        x20 = _M.lowpass(L[:, 1], fs, 20)
        sinus = e.truth["rhythm"] == "sinus"
        pr, qrs, qt, net_i, net_avf, t_prev = [], [], [], [], [], None
        for k in range(len(r) - 1):
            b = max(0, on[k] - int(0.03 * fs))
            te = _M.t_end_tangent(x20, fs, off[k], r[k + 1], np.median(x20[b:on[k]]))
            qrs.append((off[k] - on[k]) / fs * 1000)
            qt.append((te - on[k]) / fs * 1000)
            if sinus and t_prev is not None and np.isfinite(t_prev):
                pr.append((on[k] - _M.p_onset(x20, fs, on[k], earliest=t_prev + 0.02 * fs)) / fs * 1000)
            t_prev = te
            for col, out in ((0, net_i), (5, net_avf)):
                seg = L[on[k]:off[k] + 1, col] - np.median(L[b:on[k], col])
                out.append(seg.max() + seg.min())
        axis = [vclinic.frontal_axis(a, v) for a, v in zip(net_i, net_avf)]
        return {"hr": 60 / (np.mean(np.diff(r)) / fs), "pr": np.nanmedian(pr) if np.isfinite(pr).any() else np.nan,
                "qrs": np.median(qrs), "qt": np.nanmedian(qt), "axis": float(np.median(axis)),
                "net_i": net_i, "net_avf": net_avf}


    def _project_table(patients):
        rows = []
        for pid in patients:
            p = clinic.set_index("patient_id").loc[pid]
            e = vclinic.patient_ecg(clinic, pid)
            m, t = _project_measure(e), e.truth
            for name, key, true in [("kalp hızı", "hr", t["hr_bpm"]), ("PR", "pr", t["pr_ms"]), ("QRS", "qrs", t["qrs_ms"]),
                                    ("QT", "qt", t["qt_ms"]), ("eksen", "axis", t["axis_deg"])]:
                rows.append({"hasta": pid, "ritim": "sinüs" if p["ecg_rhythm"] == "sinus" else "AF",
                             "ileti": p["ecg_conduction"], "ölçüm": name,
                             "ölçülen": round(float(m[key]), 1), "gerçek": round(float(true), 1)})
        return pd.DataFrame(rows)


    def _project_qt(patients):
        rows = []
        for pid in patients:
            p = clinic.set_index("patient_id").loc[pid]
            m = _project_measure(vclinic.patient_ecg(clinic, pid))
            rows.append({"hasta": pid, "cinsiyet": "kadın" if p["sex"] == "F" else "erkek",
                         "kalp hızı": round(m["hr"], 1), "QT": round(m["qt"], 1)})
        return pd.DataFrame(rows)
_m = _project_measure(vclinic.patient_ecg(clinic, "VC-0317"))
veri = pd.DataFrame({"atım": np.arange(1, len(_m["net_i"]) + 1),
                     "D1": np.round(_m["net_i"], 3), "aVF": np.round(_m["net_avf"], 3)})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 2: Gerçek değerlere karşı on hasta

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Otomatik bir EKG ölçümünün, aralarında atriyal fibrilasyonu ya da sol dal bloğu olanların da bulunduğu on sanal klinik hastasında gerçek değerlerle ne kadar uyuştuğunu öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var. Her satır bir hastanın bir ölçümüdür: "hasta" sütunu hastanın numarasını verir; "ritim" sütununda "sinüs" ya da "AF", "ileti" sütununda "normal" ya da "LBBB" (sol dal bloğu) yazar. "ölçüm" sütunu ölçümün adını ("kalp hızı", "PR", "QRS", "QT" ya da "eksen"), "ölçülen" sütunu otomatik ölçümün değerini, "gerçek" sütunu da gerçek değeri verir. Kalp hızları dakikadaki atım sayısı, aralıklar milisaniye, eksenler derece olarak verilmiştir. Atriyal fibrilasyonda PR aralığı olmadığından bu hücreler boştur.  
> Görev: Her satır için ölçülen ile gerçek değer arasındaki farkı (ölçülen eksi gerçek) hesapla. Her ölçüm türü için ortalama mutlak farkı ve en büyük mutlak farkı, bu farkın görüldüğü hastayla birlikte yazdır. Ardından bütün farkları, her hastanın bir satırda, her ölçümün bir sütunda yer aldığı tek bir tabloda göster.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: İki özeti ve tabloyu yazdır; QT aralığının ortalama mutlak farkını milisaniye olarak "cevap" adıyla sakla.  
> Test: Tabloda on satır bulunmalı. VC-0317 numaralı hastada PR, QRS ve QT farkları yalnızca birkaç milisaniye olmalı.  

In [ ]:
#@title Proje adımı 2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["ann_samples", "ann_symbols", "clinic", "fs_mit", "project_patients", "read_physionet_annotation"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    from vclinic import measure as _M
    from vclinic.datasets import read_physionet_annotation
    clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
    ann_samples, ann_symbols = read_physionet_annotation("100", "mitdb")
    fs_mit = 360
    proje_hastalari = ["VC-0317", "VC-0079", "VC-0254", "VC-0188", "VC-0042",
                         "VC-0150", "VC-0007", "VC-0031", "VC-0018", "VC-0026"]


    def _project_measure(e):
        """Heart rate, PR, QRS, QT and frontal axis of one ECG, with the net QRS voltages of leads I and aVF in every beat."""
        L, fs = e.leads.to_numpy(), e.fs
        r = _M.detect_r_peaks(L[:, 1], fs)
        on, off = _M.qrs_bounds(L, fs, r)
        x20 = _M.lowpass(L[:, 1], fs, 20)
        sinus = e.truth["rhythm"] == "sinus"
        pr, qrs, qt, net_i, net_avf, t_prev = [], [], [], [], [], None
        for k in range(len(r) - 1):
            b = max(0, on[k] - int(0.03 * fs))
            te = _M.t_end_tangent(x20, fs, off[k], r[k + 1], np.median(x20[b:on[k]]))
            qrs.append((off[k] - on[k]) / fs * 1000)
            qt.append((te - on[k]) / fs * 1000)
            if sinus and t_prev is not None and np.isfinite(t_prev):
                pr.append((on[k] - _M.p_onset(x20, fs, on[k], earliest=t_prev + 0.02 * fs)) / fs * 1000)
            t_prev = te
            for col, out in ((0, net_i), (5, net_avf)):
                seg = L[on[k]:off[k] + 1, col] - np.median(L[b:on[k], col])
                out.append(seg.max() + seg.min())
        axis = [vclinic.frontal_axis(a, v) for a, v in zip(net_i, net_avf)]
        return {"hr": 60 / (np.mean(np.diff(r)) / fs), "pr": np.nanmedian(pr) if np.isfinite(pr).any() else np.nan,
                "qrs": np.median(qrs), "qt": np.nanmedian(qt), "axis": float(np.median(axis)),
                "net_i": net_i, "net_avf": net_avf}


    def _project_table(patients):
        rows = []
        for pid in patients:
            p = clinic.set_index("patient_id").loc[pid]
            e = vclinic.patient_ecg(clinic, pid)
            m, t = _project_measure(e), e.truth
            for name, key, true in [("kalp hızı", "hr", t["hr_bpm"]), ("PR", "pr", t["pr_ms"]), ("QRS", "qrs", t["qrs_ms"]),
                                    ("QT", "qt", t["qt_ms"]), ("eksen", "axis", t["axis_deg"])]:
                rows.append({"hasta": pid, "ritim": "sinüs" if p["ecg_rhythm"] == "sinus" else "AF",
                             "ileti": p["ecg_conduction"], "ölçüm": name,
                             "ölçülen": round(float(m[key]), 1), "gerçek": round(float(true), 1)})
        return pd.DataFrame(rows)


    def _project_qt(patients):
        rows = []
        for pid in patients:
            p = clinic.set_index("patient_id").loc[pid]
            m = _project_measure(vclinic.patient_ecg(clinic, pid))
            rows.append({"hasta": pid, "cinsiyet": "kadın" if p["sex"] == "F" else "erkek",
                         "kalp hızı": round(m["hr"], 1), "QT": round(m["qt"], 1)})
        return pd.DataFrame(rows)
# Projenin on hastası. Liste değiştirilebilir; en az ikisinde atriyal fibrilasyon ya da dal bloğu bulunmalıdır.
proje_hastalari = ["VC-0317", "VC-0079", "VC-0254", "VC-0188", "VC-0042",
                     "VC-0150", "VC-0007", "VC-0031", "VC-0018", "VC-0026"]
veri = _project_table(proje_hastalari)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 3: 100 numaralı kayıtta otuz dakika boyunca kalp hızı

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: MIT-BIH Aritmi Veri Tabanı'ndaki 100 numaralı kayıtta, yani 69 yaşında bir erkeğin 30 dakikalık ambulatuvar EKG'sinde, kalp hızının zaman içinde nasıl değiştiğini görmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var. Her satır, kardiyologların kayda koyduğu bir işarettir: "zaman" sütunu işaretin yerini saniye olarak, "etiket" sütunu da türünü verir. "N" normal atımı, "A" atriyal erken atımı, "V" ventriküler erken atımı gösterir; "+" ise ritim değişikliğini belirtir ve bir atım değildir.  
> Görev: Yalnızca atımları, yani "+" dışındaki etiketleri tut. İlk atımdan sonraki her atım için kalp hızını, 60'ın saniye cinsinden RR aralığına bölümü olarak hesapla. Bu kalp hızını dakika cinsinden zamana karşı çiz. Ardından kaydın tamamının ortalama kalp hızını, 60'ın ortalama RR aralığına bölümüyle hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Ortalama kalp hızını dakikadaki atım sayısı olarak, bir ondalık basamakla yazdır ve "cevap" adıyla sakla. Grafiği de göster.  
> Test: Tam 1 saniyelik bir RR aralığı dakikada 60 atım vermeli; ortalama kalp hızı da dakikada 60 ile 90 atım arasında olmalı.  

In [ ]:
#@title Proje adımı 3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["ann_samples", "ann_symbols", "clinic", "fs_mit", "project_patients", "read_physionet_annotation"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    from vclinic import measure as _M
    from vclinic.datasets import read_physionet_annotation
    clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
    ann_samples, ann_symbols = read_physionet_annotation("100", "mitdb")
    fs_mit = 360
    proje_hastalari = ["VC-0317", "VC-0079", "VC-0254", "VC-0188", "VC-0042",
                         "VC-0150", "VC-0007", "VC-0031", "VC-0018", "VC-0026"]


    def _project_measure(e):
        """Heart rate, PR, QRS, QT and frontal axis of one ECG, with the net QRS voltages of leads I and aVF in every beat."""
        L, fs = e.leads.to_numpy(), e.fs
        r = _M.detect_r_peaks(L[:, 1], fs)
        on, off = _M.qrs_bounds(L, fs, r)
        x20 = _M.lowpass(L[:, 1], fs, 20)
        sinus = e.truth["rhythm"] == "sinus"
        pr, qrs, qt, net_i, net_avf, t_prev = [], [], [], [], [], None
        for k in range(len(r) - 1):
            b = max(0, on[k] - int(0.03 * fs))
            te = _M.t_end_tangent(x20, fs, off[k], r[k + 1], np.median(x20[b:on[k]]))
            qrs.append((off[k] - on[k]) / fs * 1000)
            qt.append((te - on[k]) / fs * 1000)
            if sinus and t_prev is not None and np.isfinite(t_prev):
                pr.append((on[k] - _M.p_onset(x20, fs, on[k], earliest=t_prev + 0.02 * fs)) / fs * 1000)
            t_prev = te
            for col, out in ((0, net_i), (5, net_avf)):
                seg = L[on[k]:off[k] + 1, col] - np.median(L[b:on[k], col])
                out.append(seg.max() + seg.min())
        axis = [vclinic.frontal_axis(a, v) for a, v in zip(net_i, net_avf)]
        return {"hr": 60 / (np.mean(np.diff(r)) / fs), "pr": np.nanmedian(pr) if np.isfinite(pr).any() else np.nan,
                "qrs": np.median(qrs), "qt": np.nanmedian(qt), "axis": float(np.median(axis)),
                "net_i": net_i, "net_avf": net_avf}


    def _project_table(patients):
        rows = []
        for pid in patients:
            p = clinic.set_index("patient_id").loc[pid]
            e = vclinic.patient_ecg(clinic, pid)
            m, t = _project_measure(e), e.truth
            for name, key, true in [("kalp hızı", "hr", t["hr_bpm"]), ("PR", "pr", t["pr_ms"]), ("QRS", "qrs", t["qrs_ms"]),
                                    ("QT", "qt", t["qt_ms"]), ("eksen", "axis", t["axis_deg"])]:
                rows.append({"hasta": pid, "ritim": "sinüs" if p["ecg_rhythm"] == "sinus" else "AF",
                             "ileti": p["ecg_conduction"], "ölçüm": name,
                             "ölçülen": round(float(m[key]), 1), "gerçek": round(float(true), 1)})
        return pd.DataFrame(rows)


    def _project_qt(patients):
        rows = []
        for pid in patients:
            p = clinic.set_index("patient_id").loc[pid]
            m = _project_measure(vclinic.patient_ecg(clinic, pid))
            rows.append({"hasta": pid, "cinsiyet": "kadın" if p["sex"] == "F" else "erkek",
                         "kalp hızı": round(m["hr"], 1), "QT": round(m["qt"], 1)})
        return pd.DataFrame(rows)
veri = pd.DataFrame({"zaman": np.asarray(ann_samples) / fs_mit, "etiket": ann_symbols})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 4: Her hasta için Bazett mi Fridericia mı?

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Sanal klinikteki on hastada, Bazett ya da Fridericia düzeltmesini seçmenin QT aralığının uzun olup olmadığı kararını değiştirip değiştirmediğini öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var. Her satır bir hastadır: "hasta" sütunu hastanın numarasını verir; "cinsiyet" sütununda "kadın" ya da "erkek" yazar. "kalp hızı" sütunu ölçülen kalp hızını dakikadaki atım sayısı olarak, "QT" sütunu da ölçülen QT aralığını milisaniye olarak verir.  
> Görev: Her hasta için RR aralığını saniye olarak, 60'ın kalp hızına bölümüyle hesapla. QTc'yi Bazett formülüyle (QT bölü RR'nin karekökü) ve Fridericia formülüyle (QT bölü RR'nin küp kökü) bul. Kadınlarda 460 ms ya da daha uzun, erkeklerde 450 ms ya da daha uzun bir QTc'yi uzun kabul et. İki formülün farklı karara götürdüğü hastaları listele.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: İki düzeltmeyi ve iki kararı içeren tabloyu ve kararları uyuşmayan hastaların listesini yazdır. Kararları farklı olan hastaların sayısını "cevap" adıyla sakla.  
> Test: Dakikada yaklaşık 85 atımlık kalp hızıyla bir kadın olan VC-0317, Bazett formülüne göre uzun, Fridericia formülüne göre normal çıkmalı.  

In [ ]:
#@title Proje adımı 4 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals():
    print("0. bölüm bu oturumda henüz çalışmadı. Şimdi çalışıyor ve ders dosyalarını, paketleri ve beklenen cevapları hazırlıyor; bu bir dakika kadar sürebilir.")
    _catch_up(1, "0. bölüm hazır.")
if any(_n not in globals() for _n in ["ann_samples", "ann_symbols", "clinic", "fs_mit", "project_patients", "read_physionet_annotation"]):
    VERI_KAYNAGI = globals().get("VERI_KAYNAGI", "web")
    from vclinic import measure as _M
    from vclinic.datasets import read_physionet_annotation
    clinic = vclinic.make_cohort(n=500, seed=RANDOM_STATE)
    ann_samples, ann_symbols = read_physionet_annotation("100", "mitdb")
    fs_mit = 360
    proje_hastalari = ["VC-0317", "VC-0079", "VC-0254", "VC-0188", "VC-0042",
                         "VC-0150", "VC-0007", "VC-0031", "VC-0018", "VC-0026"]


    def _project_measure(e):
        """Heart rate, PR, QRS, QT and frontal axis of one ECG, with the net QRS voltages of leads I and aVF in every beat."""
        L, fs = e.leads.to_numpy(), e.fs
        r = _M.detect_r_peaks(L[:, 1], fs)
        on, off = _M.qrs_bounds(L, fs, r)
        x20 = _M.lowpass(L[:, 1], fs, 20)
        sinus = e.truth["rhythm"] == "sinus"
        pr, qrs, qt, net_i, net_avf, t_prev = [], [], [], [], [], None
        for k in range(len(r) - 1):
            b = max(0, on[k] - int(0.03 * fs))
            te = _M.t_end_tangent(x20, fs, off[k], r[k + 1], np.median(x20[b:on[k]]))
            qrs.append((off[k] - on[k]) / fs * 1000)
            qt.append((te - on[k]) / fs * 1000)
            if sinus and t_prev is not None and np.isfinite(t_prev):
                pr.append((on[k] - _M.p_onset(x20, fs, on[k], earliest=t_prev + 0.02 * fs)) / fs * 1000)
            t_prev = te
            for col, out in ((0, net_i), (5, net_avf)):
                seg = L[on[k]:off[k] + 1, col] - np.median(L[b:on[k], col])
                out.append(seg.max() + seg.min())
        axis = [vclinic.frontal_axis(a, v) for a, v in zip(net_i, net_avf)]
        return {"hr": 60 / (np.mean(np.diff(r)) / fs), "pr": np.nanmedian(pr) if np.isfinite(pr).any() else np.nan,
                "qrs": np.median(qrs), "qt": np.nanmedian(qt), "axis": float(np.median(axis)),
                "net_i": net_i, "net_avf": net_avf}


    def _project_table(patients):
        rows = []
        for pid in patients:
            p = clinic.set_index("patient_id").loc[pid]
            e = vclinic.patient_ecg(clinic, pid)
            m, t = _project_measure(e), e.truth
            for name, key, true in [("kalp hızı", "hr", t["hr_bpm"]), ("PR", "pr", t["pr_ms"]), ("QRS", "qrs", t["qrs_ms"]),
                                    ("QT", "qt", t["qt_ms"]), ("eksen", "axis", t["axis_deg"])]:
                rows.append({"hasta": pid, "ritim": "sinüs" if p["ecg_rhythm"] == "sinus" else "AF",
                             "ileti": p["ecg_conduction"], "ölçüm": name,
                             "ölçülen": round(float(m[key]), 1), "gerçek": round(float(true), 1)})
        return pd.DataFrame(rows)


    def _project_qt(patients):
        rows = []
        for pid in patients:
            p = clinic.set_index("patient_id").loc[pid]
            m = _project_measure(vclinic.patient_ecg(clinic, pid))
            rows.append({"hasta": pid, "cinsiyet": "kadın" if p["sex"] == "F" else "erkek",
                         "kalp hızı": round(m["hr"], 1), "QT": round(m["qt"], 1)})
        return pd.DataFrame(rows)
veri = _project_qt(proje_hastalari)
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Python ile başlangıç: Başka bir hasta, başka bir derivasyon

Aşağıdaki hücre sanal klinikteki bir hastayı `measure_ecg` ile ölçer ve sonucu gerçek değerlerle karşılaştırır. Önce hazırlık hücresini çalıştırın; bu hücre, ölçüm hücresinin ihtiyaç duyduğu durak kodlarını çalıştırır. Ardından `PATIENT` ve `LEAD` değerlerini değiştirin ve hücreyi yeniden çalıştırın. Bakmaya değer hastalar şunlardır: VC-0183 (uzun QT aralığı), VC-0018 (sol dal bloğu), VC-0207 (kayıtta 218 ms'lik bir PR aralığı) ve `ecg_rhythm` değeri `AF` olan herhangi bir hasta. `clinic[clinic["stemi_territory"] != "none"]`, 5. haftanın konusu olan akut ST yükselmeli miyokard enfarktüsü (STEMI) tanılı hastaları listeler.

In [ ]:
#@title Python başlangıcının hazırlığı: Bu hücreyi bir kez çalıştırın (aşağıdaki hücrenin ihtiyaç duyduğu durak kodlarını çalıştırır)
def _catch_up(upto, done, name="NB01_python_ve_ekg_temelleri.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-01/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print(done)


if "teach" not in globals() or any(_n not in globals() for _n in ["clinic", "measure_ecg"]):
    _catch_up(305, "0. bölüm ve burada gereken önceki kodlar çalıştı.")
print("Hazır: Aşağıdaki hücreyi çalıştırın.")

In [ ]:
PATIENT = "VC-0183"
LEAD = "II"

e = vclinic.patient_ecg(clinic, PATIENT)
row = clinic[clinic["patient_id"] == PATIENT].iloc[0]
print(PATIENT, row["age"], row["sex"], row["main_diagnosis"], row["ecg_rhythm"], row["ecg_conduction"])
m = measure_ecg(e.leads.to_numpy(), e.fs, lead=LEAD, sinus=e.truth["rhythm"] == "sinus")
keys = ["hr_bpm", "pr_ms", "qrs_ms", "qt_ms", "qtc_bazett_ms", "qtc_fridericia_ms"]
display(pd.DataFrame({"ölçülen": [m[k] for k in keys],
                      "gerçek": [e.truth[k] for k in keys]}, index=keys).round(1))
fig = teach.plot_12lead(e.time, e.leads, title=PATIENT)
plt.show()

**Haftalık projenin sonu.** [Ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/ders.html#weekly-project): «Haftalık proje» bölümü raporu ve teslimi anlatır.

## Alanlara göre uygulama görevleri

| Alan | Görev |
|---|---|
| Elektrofizyoloji | VC-0183'ün EKG'sini dakikada 50 ile 110 atım arasındaki kalp hızlarında üretin (`e.params` içindeki `hr_bpm` değerini değiştirin ve `qt_ms` değerini $RR^{1/3}$ ile ölçekleyin). Bazett ve Fridericia QTc değerlerini kalp hızına göre çizin ve bir QT izlem protokolünde hangisinin raporlanacağına karar verin. |
| Girişimsel kardiyoloji | STEMI bölgesi olan hastaları listeleyin, 12 derivasyonlu EKG'lerini çizin ve her bölge için ST yükselmesi olan derivasyonları adlandırın. Sonucu, bölgenin işaret ettiği sorumlu arterle karşılaştırın. |
| Kalp yetersizliği | Klinikteki HFrEF hastalarıyla kalp hastalığı olmayan hastalar arasında 5 dakikalık kayıtların SDNN ve RMSSD değerlerini karşılaştırın. |
| Kardiyo-onkoloji | QT'yi uzatan bir tedavi alan bir hasta için, farklı kalp hızlarındaki seri EKG'lerle bir QTc izlem tablosu oluşturun ve QTc'nin yerel protokolünüzdeki eşiği aştığı vizitleri işaretleyin. |
| Pediatrik kardiyoloji | 2009 önerileri çocuklarda daha düşük QRS eşikleri kullanmaktadır. Kendi yazacağınız bir fonksiyonda geniş QRS eşiğini değiştirin ve bir çocuğun EKG'sinde başka nelerin değişeceğini tartışın. |
| Spor kardiyolojisi | Kalp hızı dakikada 45 atım olan bir istirahat EKG'si üretin ve iki formülün verdiği QTc değerlerini karşılaştırın: Hangisi bir sporcuyu anormal olarak etiketleyebilir? |
| Yoğun bakım | `noise="clinic"` ile 5 dakikalık bir kayıt üretin ve R tepesi tespit algoritmasının Se ve PPV değerlerini gerçek atım zamanlarına göre ölçün. |

## Jupyter Notebook'tan sonra

[Etkileşimli labı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-01/lab.html) açın ve *Kendini sına* sekmesindeki öz değerlendirmeyi yapın. Ardından *Düşün ve dışa aktar* sekmesinde yansıtma metnini yazın ve öğrenme günlüğünü dışa aktarın. [Hafta sayfasında](README.md) anlatılan 1. haftanın haftalık projesi, «Durak 2 · Hazır uygulama» bölümündeki `measure_ecg` fonksiyonundan başlar.

<div align="center">

---

**Prof. Dr. Utku Köse**<br>
Profesör, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi, Isparta, Türkiye<br>
Kurucu Müdür, Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM), Süleyman Demirel Üniversitesi<br>
Bilgisayar Bilimleri Anabilim Dalı Başkanı, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi<br>
Diğer kurumlar: University of North Dakota (ABD), Universidad Panamericana (Meksiko, Meksika), Vel Tech University (Chennai, Hindistan)<br>
IEEE Kıdemli Üyesi, ACM Profesyonel Üyesi

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı, Türk Kardiyoloji Derneği. İlk sürüm: Kasım 2026. İçerik CC BY 4.0, kod MIT.*

</div>